# DCIC-Flix26 — Perfilado de calidad del dataset

Análisis exploratorio de las colecciones `movies` y `reviews` provistas por la cátedra.
**No modifica** los archivos de `dataset/`: solo los lee.

Los archivos son NDJSON (un documento por línea) en **MongoDB Extended JSON** (`{"$oid": ...}`, `{"$date": ...}`).
Se parsean con `bson.json_util`, la misma semántica que usa `mongoimport`, para ver los tipos
exactamente como quedarían en Mongo: una fecha válida llega como `datetime`, una fecha mal cargada sigue siendo `str`.

## 0. Setup y carga

In [1]:
import gzip
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from bson import json_util

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

# El notebook vive en analysis/; el dataset está en ../dataset
DATASET_DIR = Path("..") / "dataset"
assert DATASET_DIR.exists(), f"No se encontró {DATASET_DIR.resolve()}"

In [2]:
def loadCollection(name: str) -> list[dict]:
    """Lee un .json.gz NDJSON en Extended JSON y devuelve la lista de documentos."""
    with gzip.open(DATASET_DIR / f"{name}.json.gz", "rt", encoding="utf-8") as f:
        return [json_util.loads(line) for line in f if line.strip()]

movies = loadCollection("movies")
reviews = loadCollection("reviews")
print(f"movies:  {len(movies):>6} documentos")
print(f"reviews: {len(reviews):>6} documentos")

movies:   19842 documentos
reviews:  61488 documentos


In [3]:
# Conteo de wrappers de Extended JSON sobre el texto crudo (antes de parsear).
# Sirve para cruzar contra los tipos parseados: si hay N documentos pero menos de N "$date",
# esas fechas llegaron sin wrapper y Mongo las guardaría como string/número, no como Date.
import re

def countWrappers(name: str) -> pd.Series:
    counter = Counter()
    with gzip.open(DATASET_DIR / f"{name}.json.gz", "rt", encoding="utf-8") as f:
        for line in f:
            counter.update(re.findall(r'"(\$(?:oid|date|numberLong|numberInt|numberDouble|numberDecimal))"', line))
    return pd.Series(counter, name=name)

pd.DataFrame([countWrappers("movies"), countWrappers("reviews")]).fillna(0).astype(int).assign(
    documentos=[len(movies), len(reviews)]
)

,$oid,$date,$numberLong,documentos
movies,19842,51387,2107,19842
reviews,122976,60578,0,61488


## 1. Perfil de campos

Se recorre cada documento recursivamente y se registra cada *path*:
- `imdb.rating` → campo anidado dentro de un subdocumento.
- `genres[]` → elementos de una lista (los porcentajes se calculan sobre el total de **elementos**, no de documentos).

Para cada path se distingue:
- **faltante**: el documento no tiene la clave (o no tiene el padre).
- **null**: la clave existe con valor `null`.
- **vacío**: string vacío/solo espacios, o lista/dict vacío.

Los ejemplos se toman **por tipo** (hasta 2 por cada tipo), para que los tipos minoritarios, que suelen ser las anomalías, no queden ocultos.

In [4]:
def walk(value, path, out):
    """Agrega (path, valor) a out para cada campo, subcampo y elemento de lista."""
    if isinstance(value, dict):
        for k, v in value.items():
            p = f"{path}.{k}" if path else k
            out.append((p, v))
            walk(v, p, out)
    elif isinstance(value, list):
        for e in value:
            out.append((f"{path}[]", e))
            walk(e, f"{path}[]", out)


def isEmpty(v) -> bool:
    return (isinstance(v, str) and v.strip() == "") or (isinstance(v, (list, dict)) and len(v) == 0)


def profile(docs: list[dict]) -> pd.DataFrame:
    types = defaultdict(Counter)
    docsWithField = Counter()
    nulls, empties = Counter(), Counter()
    examples = defaultdict(lambda: defaultdict(list))

    for doc in docs:
        pairs = []
        walk(doc, "", pairs)
        for p in {p for p, _ in pairs if not p.endswith("[]")}:
            docsWithField[p] += 1
        for p, v in pairs:
            t = type(v).__name__
            types[p][t] += 1
            nulls[p] += v is None
            empties[p] += isEmpty(v)
            ex = examples[p][t]
            if len(ex) < 2 and not isinstance(v, (dict, list)) and v not in ex:
                ex.append(v)

    n = len(docs)
    rows = []
    for p in sorted(types):
        isElement = p.endswith("[]")
        total = sum(types[p].values())
        base = total if isElement else n
        rows.append({
            "campo": p,
            "tipos": ", ".join(f"{t}: {c}" for t, c in types[p].most_common()),
            "% faltante": None if isElement else round(100 * (n - docsWithField[p]) / n, 2),
            "% null": round(100 * nulls[p] / base, 2),
            "% vacío": round(100 * empties[p] / base, 2),
            "ejemplos": "; ".join(
                f"[{t}] " + " | ".join(repr(x)[:45] for x in xs)
                for t, xs in examples[p].items() if xs
            ),
        })
    return pd.DataFrame(rows).set_index("campo")

### 1.1 `movies`

In [5]:
moviesProfile = profile(movies)
moviesProfile

,tipos,% faltante,% null,% vacío,ejemplos
campo,,,,,
_id,ObjectId: 19842,0.00,0.00,0.00,[ObjectId] ObjectId('573a13a4f29313caabd10527') | ObjectId('573a13cff29313caabd88e6c')
awards,dict: 19842,0.00,0.00,0.00,
awards.nominations,int: 19842,0.00,0.00,0.00,[int] 7 | 1
awards.text,str: 19842,0.00,0.00,0.00,[str] '1 win & 7 nominations.' | '1 nomination.'
awards.wins,int: 19842,0.00,0.00,0.00,[int] 1 | 0
cast,list: 19638,1.03,0.00,0.00,
cast[],str: 77449,NaN,0.00,0.00,[str] 'Nis Bank-Mikkelsen' | 'Alan Rickman'
countries,list: 19834,0.04,0.00,0.00,
countries[],str: 28453,NaN,0.00,0.00,[str] 'Germany' | 'Ireland'


### 1.2 `reviews`

In [6]:
reviewsProfile = profile(reviews)
reviewsProfile

,tipos,% faltante,% null,% vacío,ejemplos
campo,,,,,
_id,ObjectId: 61488,0.0,0.00,0.00,[ObjectId] ObjectId('6af3753e68313c9a2da138a5') | ObjectId('6af3753e68313c9a2da0c6d6')
comment,str: 61488,0.0,0.00,0.32,"[str] 'A very compelling watch with standout lead p | ""Not bad, but ultimately didn't leave a lasti"
date,"datetime: 60578, str: 733, int: 177",0.0,0.00,0.00,"[datetime] datetime.datetime(2024, 3, 10, 0, 1, 43) | datetime.datetime(2025, 9, 1, 6, 57, 51); [str] '06/04/2024' |..."
movie_id,ObjectId: 61488,0.0,0.00,0.00,[ObjectId] ObjectId('573a1397f29313caabce8475') | ObjectId('573a13a5f29313caabd13b4b')
rating,"int: 59687, NoneType: 928, str: 873",0.0,1.51,0.00,[int] 4 | 3; [NoneType] None; [str] '4' | '3'
user_email,str: 61488,0.0,0.00,0.75,[str] 'leo.roberts551@proton.me' | 'casey.adams980@yahoo.com'
user_id,str: 61488,0.0,0.00,0.37,[str] 'usr_0937' | 'usr_0177'
user_name,str: 61488,0.0,0.00,0.00,[str] 'Leo Roberts' | 'Casey Adams'


### 1.3 Campos con más de un tipo
Un campo con tipos mezclados es candidato directo a anomalía (o a conversión en la limpieza).

In [7]:
def mixedTypes(prof: pd.DataFrame) -> pd.DataFrame:
    # NoneType no cuenta como tipo distinto: se reporta aparte como null
    nonNull = prof["tipos"].str.replace(r"(, )?NoneType: \d+", "", regex=True).str.strip(", ")
    return prof[nonNull.str.contains(",")][["tipos", "% null"]]

pd.concat({"movies": mixedTypes(moviesProfile), "reviews": mixedTypes(reviewsProfile)})

tipos  % null
        campo                                                              
movies  genres                                list: 19421, str: 378    0.00
        imdb.rating               float: 17797, int: 1892, str: 153    0.00
        imdb.votes                             int: 19418, str: 424    0.00
        runtime                  int: 19271, str: 255, NoneType: 91    0.46
        tomatoes.critic.rating              float: 10118, int: 1282    0.00
        tomatoes.viewer.rating                float: 5396, int: 661    0.00
        year                    int: 19408, str: 321, NoneType: 113    0.57
reviews date                    datetime: 60578, str: 733, int: 177    0.00
        rating                  int: 59687, NoneType: 928, str: 873    1.51

## 2. Anomalías conocidas — `movies` (Paso 3a)

Se arma un DataFrame plano con los campos relevantes **sin convertir tipos** (columnas `object`): la idea es ver
los valores tal cual vienen. Los campos ausentes se marcan con un centinela `MISSING`, distinto de `None` (null).

In [8]:
import unicodedata


class _Missing:
    def __repr__(self):
        return "<faltante>"

MISSING = _Missing()


def pattern(value) -> str:
    """Forma de un valor: cada secuencia de dígitos se reemplaza por N ('2015è' -> 'Nè', '2h 1m' -> 'Nh Nm').
    Los no-string se muestran como <tipo>."""
    if isinstance(value, str):
        return repr(re.sub(r"\d+", "N", value))
    return "<faltante>" if value is MISSING else f"<{type(value).__name__}>"


def patternTable(series: pd.Series, nExamples: int = 4) -> pd.DataFrame:
    """Cuenta cada patrón distinto y muestra ejemplos reales de cada uno."""
    pats = series.map(pattern)
    rows = []
    for pat, count in pats.value_counts().items():
        values = series[pats == pat]
        uniq = list(dict.fromkeys(map(repr, values)))[:nExamples]
        rows.append({"patrón": pat, "casos": count, "ejemplos": " · ".join(uniq)})
    return pd.DataFrame(rows).set_index("patrón")


def sample(df: pd.DataFrame, n: int = 8) -> pd.DataFrame:
    """Muestra reproducible (random_state fijo) de hasta n filas."""
    return df.sample(min(n, len(df)), random_state=42) if len(df) else df


moviesDf = pd.DataFrame({
    "_id": [str(m["_id"]) for m in movies],
    "title": [m["title"] for m in movies],
    "year": [m.get("year", MISSING) for m in movies],
    "runtime": [m.get("runtime", MISSING) for m in movies],
    "genres": [m.get("genres", MISSING) for m in movies],
    "imdbId": [m["imdb"]["id"] for m in movies],
    "imdbRating": [m["imdb"].get("rating", MISSING) for m in movies],
    "imdbVotes": [m["imdb"].get("votes", MISSING) for m in movies],
    "type": [m.get("type", MISSING) for m in movies],
    "directors": [m.get("directors", MISSING) for m in movies],
}, dtype=object)
moviesDf.head(3)

,_id,title,year,runtime,genres,imdbId,imdbRating,imdbVotes,type,directors
0,573a13a4f29313caabd10527,A Fish Tale,2000,80,"[Animation, Adventure, Comedy]",168856,5.8,3760,movie,"[Stefan Fjeldmark, Michael Hegner, Greg Manwaring]"
1,573a13cff29313caabd88e6c,Hunky Dory,2011,110,"[Comedy, Drama, Music]",1727300,6,742,movie,[Marc Evans]
2,573a1397f29313caabce8347,Blade Runner,1982,117,"[Sci-Fi, Thriller]",83658,8.2,419589,movie,[Ridley Scott]


### 2.1 Duplicados

Dos criterios candidatos:
- **A. `imdb.id`**: identificador externo, una película = un id.
- **B. título normalizado + año**: lo que haríamos si no confiáramos en el id.

Para B el título se normaliza (NFKC, espacios colapsados, minúsculas). El año se prueba de dos formas:
**crudo** (tal cual viene) y **rescatado** (se extrae el número de `'circa 1994'`, `'1994è'`, `-1994`, etc.).
`rescueYear` es solo para comparar criterios: **no** es la regla de limpieza, eso se define en 2.2.

In [9]:
def normTitle(t: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", t)).strip().casefold()


def rescueYear(y):
    if isinstance(y, int):
        return abs(y)
    if isinstance(y, str):
        found = re.search(r"\d{4}", y)
        return int(found.group()) if found else None
    return None


moviesDf["titleNorm"] = moviesDf["title"].map(normTitle)
moviesDf["yearRaw"] = moviesDf["year"].map(repr)  # repr para poder agrupar int, str y None juntos
moviesDf["yearRescued"] = moviesDf["year"].map(rescueYear)


def dupSummary(keys) -> dict:
    sizes = moviesDf.groupby(keys, dropna=False, sort=False).size()
    groups = sizes[sizes > 1]
    return {
        "grupos duplicados": len(groups),
        "documentos en grupos": int(groups.sum()),
        "documentos sobrantes": int((groups - 1).sum()),
        "tamaño máx. de grupo": int(groups.max()),
    }


pd.DataFrame({
    "A. imdb.id": dupSummary("imdbId"),
    "B1. título + año crudo": dupSummary(["titleNorm", "yearRaw"]),
    "B2. título + año rescatado": dupSummary(["titleNorm", "yearRescued"]),
})

,A. imdb.id,B1. título + año crudo,B2. título + año rescatado
grupos duplicados,651,640,667
documentos en grupos,1348,1323,1379
documentos sobrantes,697,683,712
tamaño máx. de grupo,5,5,5


#### ¿En qué difieren los criterios?
1. Grupos por `imdb.id` que **B no detecta** (mismo id, distinto título/año).
2. Grupos por título+año que tienen **más de un `imdb.id`**: ¿son la misma película o películas distintas homónimas?
   Se muestran los directores para juzgarlo.

In [10]:
dupA = moviesDf[moviesDf.duplicated("imdbId", keep=False)]
keysPerId = dupA.groupby("imdbId").apply(
    lambda g: g[["titleNorm", "yearRaw"]].drop_duplicates().shape[0], include_groups=False
)
idsMissedByB1 = keysPerId[keysPerId > 1].index
print(f"Grupos por imdb.id que B1 (año crudo) separa: {len(idsMissedByB1)}")
dupA[dupA["imdbId"].isin(idsMissedByB1)].sort_values("imdbId")[["imdbId", "title", "year"]].head(20)

Grupos por imdb.id que B1 (año crudo) separa: 37


,imdbId,title,year
19425,67919,The Emigrants,1971
19258,67919,The Emigrants,1971
8365,67919,The Emigrants,1971è
4835,67919,The Emigrants,1971
3182,67919,The Emigrants,1971
16486,74695,Cross of Iron,1977
9857,74695,The Cross of Iron,1977
16279,93995,Slugs: The Movie,circa 1988
6293,93995,Slugs: The Movie,1988
10856,94625,Akira,1988


In [11]:
dupB2 = moviesDf[moviesDf.duplicated(["titleNorm", "yearRescued"], keep=False)]
idsPerKey = dupB2.groupby(["titleNorm", "yearRescued"])["imdbId"].nunique()
homonyms = idsPerKey[idsPerKey > 1]
print(f"Grupos título+año con más de un imdb.id: {len(homonyms)}")
(dupB2.set_index(["titleNorm", "yearRescued"]).loc[homonyms.index]
      [["imdbId", "title", "directors", "runtime"]].drop_duplicates(subset=["imdbId"]))

Grupos título+año con más de un imdb.id: 25


imdbId                            title                                                  directors     runtime
titleNorm                       yearRescued                                                                                                                 
1                               2013.0       2518788                                1                                             [Paul Crowder]         112
                                2013.0       2395385                                1                                           [Dennis Iliadis]          95
a better life                   2011.0       2027265                    A Better Life                                              [Cèdric Kahn]         110
                                2011.0       1554091                    A Better Life                                              [Chris Weitz]          98
adrift                          2009.0       1499639                           Adrift                                          [Bui Thac Chuyèn]         110
                                2009.0       1016321                           Adrift                                            [Heitor Dhalia]          97
avenged                         2013.0       2585072                          Avenged                                            [Donovan Marsh]          96
                                2013.0       2378453                          Avenged                                         [Michael S. Ojeda]          95
black rain                      1989.0         96933                       Black Rain                                             [Ridley Scott]         125
                                1989.0         97694                       Black Rain                                           [Shèhei Imamura]         123
black sheep                     2006.0        779982                      Black Sheep                                            [Jonathan King]          87
                                2006.0        832517                      Black Sheep                                              [Oliver Rihs]          94
casanova                        2005.0        402894                         Casanova                                          [Lasse Hallstrèm]         112
                                2005.0        427042                         Casanova                                                 <faltante>         174
confessions of a dangerous mind 2002.0        270288  Confessions of a Dangerous Mind                                           [George Clooney]         113
                                2002.0        290538  Confessions of a Dangerous Mind                                           [George Clooney]         113
darling                         2007.0        881233                          Darling                                       [Christine Carrière]          93
                                2007.0        875581                          Darling                                              [Johan Kling]          94
divide and conquer              1943.0        182989               Divide and Conquer                              [Frank Capra, Anatole Litvak]          57
                                1943.0        173769               Divide and Conquer                              [Frank Capra, Anatole Litvak]          57
emma                            1996.0        118308                             Emma                                        [Diarmuid Lawrence]         107
                                1996.0        116191                             Emma                                          [Douglas McGrath]         120
eros                            2004.0        377059                             Eros  [Michelangelo Antonioni, Steven Soderbergh, Kar Wai Wong]         104
                                2004.0        343663                             Eros  [Michelangelo Antonioni, Steven So

**¿Se escapa algún duplicado con `imdb.id` distinto?** Para cada grupo título+año con varios ids se compara
directores y duración: si coinciden, es probablemente la misma película cargada con dos ids (falso negativo del criterio A).

In [12]:
def directorsKey(d):
    return tuple(sorted(d)) if isinstance(d, list) else ()

rows = []
for (title, year), group in dupB2.groupby(["titleNorm", "yearRescued"]):
    reps = group.drop_duplicates(subset="imdbId")
    if len(reps) < 2:
        continue
    sameDirectors = reps["directors"].map(directorsKey).nunique() == 1
    sameRuntime = reps["runtime"].map(repr).nunique() == 1
    rows.append({"título": title, "año": int(year), "imdb.ids": list(reps["imdbId"]),
                 "mismos directores": sameDirectors, "misma duración": sameRuntime,
                 "veredicto": "probable misma película" if sameDirectors and sameRuntime else "homónimas distintas"})

homonymCheck = pd.DataFrame(rows).sort_values("veredicto", ascending=False)
print(homonymCheck["veredicto"].value_counts().to_string())
homonymCheck

veredicto
homónimas distintas        21
probable misma película     4


,título,año,imdb.ids,mismos directores,misma duración,veredicto
22,the kingdom,1994,"[108906, 127392]",True,True,probable misma película
7,confessions of a dangerous mind,2002,"[270288, 290538]",True,True,probable misma película
9,divide and conquer,1943,"[182989, 173769]",True,True,probable misma película
11,eros,2004,"[377059, 343663]",True,True,probable misma película
0,1,2013,"[2518788, 2395385]",False,False,homónimas distintas
14,grace,2014,"[2963344, 3068192]",False,False,homónimas distintas
23,the suspect,2013,"[3404158, 2073121]",False,False,homónimas distintas
21,the girl,2012,"[1127883, 2132485]",False,False,homónimas distintas
20,the castle,1997,"[118826, 120075]",False,False,homónimas distintas
19,soldier,1998,"[120157, 211634]",False,False,homónimas distintas


#### ¿Qué campos cambian entre copias del mismo `imdb.id`?
Se comparan todas las hojas (paths sin listas expandidas) de cada grupo. Un campo "difiere" en un grupo si
alguna copia tiene un valor distinto (o no lo tiene).

In [13]:
def leaves(doc: dict) -> dict:
    """Aplana un documento a {path: valor} usando walk(); las listas se comparan enteras."""
    pairs = []
    walk(doc, "", pairs)
    return {p: v for p, v in pairs if "[]" not in p and not isinstance(v, dict)}


docsById = defaultdict(list)
for m in movies:
    docsById[m["imdb"]["id"]].append(m)
dupGroups = {k: g for k, g in docsById.items() if len(g) > 1}

fieldDiff = Counter()
for group in dupGroups.values():
    flats = [leaves(d) for d in group]
    for path in set().union(*flats):
        values = {json_util.dumps(f.get(path, "<faltante>")) for f in flats}
        if len(values) > 1:
            fieldDiff[path] += 1

(pd.Series(fieldDiff, name="grupos donde difiere").sort_values(ascending=False)
   .to_frame().assign(**{"% de grupos": lambda d: (100 * d.iloc[:, 0] / len(dupGroups)).round(1)}))

,grupos donde difiere,% de grupos
_id,651,100.0
lastupdated,615,94.5
imdb.votes,558,85.7
num_mflix_comments,434,66.7
imdb.rating,48,7.4
tomatoes.lastUpdated,48,7.4
title,39,6.0
year,36,5.5
genres,33,5.1
tomatoes.dvd,25,3.8


In [14]:
def showGroupDiff(imdbId: int) -> pd.DataFrame:
    """Muestra lado a lado SOLO los campos que difieren entre las copias de un imdb.id."""
    flats = [leaves(d) for d in docsById[imdbId]]
    paths = sorted(set().union(*flats))
    differing = [p for p in paths if len({json_util.dumps(f.get(p, "<faltante>")) for f in flats}) > 1]
    table = pd.DataFrame({str(d["_id"]): [repr(f.get(p, MISSING)) for p in differing]
                          for d, f in zip(docsById[imdbId], flats)}, index=differing)
    table.index.name = f"imdb.id={imdbId} · {docsById[imdbId][0]['title'].strip()!r}"
    return table


# Ejemplos: un grupo "típico", uno con año anómalo, uno con título con espacios y el grupo más grande
largest = max(dupGroups, key=lambda k: len(dupGroups[k]))
picked = [168856, *list(idsMissedByB1[:2]), largest]
picked += [k for k, g in dupGroups.items() if len({d["title"] for d in g}) > 1][:2]
for imdbId in dict.fromkeys(picked):
    display(showGroupDiff(imdbId))

,573a13a4f29313caabd10527,573a139ef29313caabcfc35e
imdb.id=168856 · 'A Fish Tale',,
_id,ObjectId('573a13a4f29313caabd10527'),ObjectId('573a139ef29313caabcfc35e')
imdb.votes,3760,3758
lastupdated,'2015-08-30 23:27:11.263000000','2015-08-30 00:04:41.663000000'
num_mflix_comments,1,<faltante>


,573a13aff29313caabd3139a,573a13aff29313caabd313a2,573a13aff29313caabd313a4,573a1396f29313caabce47dc,573a13aff29313caabd3139b
imdb.id=67919 · 'The Emigrants',,,,,
_id,ObjectId('573a13aff29313caabd3139a'),ObjectId('573a13aff29313caabd313a2'),ObjectId('573a13aff29313caabd313a4'),ObjectId('573a1396f29313caabce47dc'),ObjectId('573a13aff29313caabd3139b')
imdb.votes,3144,3144,3144,3162,3144
lastupdated,'2015-09-02 07:10:04.100000000','2015-09-02 07:10:02.930000000','2015-09-02 07:10:04.930000000','2015-09-17 04:41:11.943000000','2015-09-02 07:10:06.240000000'
num_mflix_comments,<faltante>,1,<faltante>,2,<faltante>
year,1971,1971,'1971è',1971,1971


,573a1397f29313caabce612d,573a1397f29313caabce65ab
imdb.id=74695 · 'The Cross of Iron',,
_id,ObjectId('573a1397f29313caabce612d'),ObjectId('573a1397f29313caabce65ab')
imdb.votes,19328,19404
lastupdated,'2015-08-21 00:02:40.157000000','2015-09-12 02:50:48.563000000'
num_mflix_comments,<faltante>,1
poster,'https://m.media-amazon.com/images/M/MV5BZGM3MWJjZTUtZmYyYi00NWVkLTgzN2ItM2U4ZmExZjIzYjVmXkEyXkFqcGdeQXVyMjI4MjA5MzA...,'https://m.media-amazon.com/images/M/MV5BZDY4ZjY1MmYtYmQzZC00Y2ZmLWFhOGYtODlhMzA4NjU2NTdjXkEyXkFqcGdeQXVyMjI4MjA5MzA...
title,'The Cross of Iron','Cross of Iron'


,573a13d1f29313caabd9056f,573a13d2f29313caabd9145b,573a13d2f29313caabd910ca,573a13d2f29313caabd90c30,573a13d2f29313caabd910c9
imdb.id=1826550 · 'Mad As Hell',,,,,
_id,ObjectId('573a13d1f29313caabd9056f'),ObjectId('573a13d2f29313caabd9145b'),ObjectId('573a13d2f29313caabd910ca'),ObjectId('573a13d2f29313caabd90c30'),ObjectId('573a13d2f29313caabd910c9')
imdb.rating,7,7.1,7.1,7.1,7.1
imdb.votes,271,263,263,263,263
lastupdated,'2015-09-16 00:43:43.073000000','2015-09-13 01:09:39.997000000','2015-09-13 00:32:38.167000000','2015-09-12 23:44:37.547000000','2015-09-13 00:32:36.870000000'
num_mflix_comments,1,1,<faltante>,2,<faltante>
year,2014,None,2014,2014,2014


,573a13a3f29313caabd0e114,573a139af29313caabcf0d04
imdb.id=120185 · 'Spice World',,
_id,ObjectId('573a13a3f29313caabd0e114'),ObjectId('573a139af29313caabcf0d04')
imdb.votes,27269,27229
lastupdated,'2015-08-30 22:43:54.450000000','2015-08-20 00:51:57.283000000'
num_mflix_comments,<faltante>,2
title,'Spice World','Spice World '


,573a139af29313caabcef98c,573a1396f29313caabce42f0
imdb.id=66580 · 'Woodstock',,
_id,ObjectId('573a139af29313caabcef98c'),ObjectId('573a1396f29313caabce42f0')
genres,'Documentary / History / Music',"['Documentary', 'History', 'Music']"
imdb.votes,11824,11842
lastupdated,'2015-08-29 07:14:58.170000000','2015-09-03 00:36:36.977000000'
num_mflix_comments,<faltante>,1
title,'Woodstock','Woodstock '


### 2.2 Año (`year`)
Patrones de todos los valores no enteros, y análisis de los enteros fuera de rango.

In [15]:
patternTable(moviesDf.loc[moviesDf["year"].map(lambda y: not isinstance(y, int)), "year"])

,casos,ejemplos
patrón,,
'Nè',123,'1981è' · '2004è' · '1993è' · '2001è'
<NoneType>,113,None
'circa N',104,'circa 1934' · 'circa 2002' · 'circa 1928' · 'circa 1999'
'N',90,'1989' · '2002' · '1987' · '1990'
'NèN',4,'2006è2007' · '2006è2012' · '1994è1998'


In [16]:
yearInt = moviesDf[moviesDf["year"].map(lambda y: isinstance(y, int))]
negYear = yearInt[yearInt["year"] < 0]
print(f"Años enteros: min={yearInt['year'].min()} max={yearInt['year'].max()}")
print(f"Años negativos: {len(negYear)}")
print(f"Positivos fuera de [1880, 2026]: {((yearInt['year'] > 0) & ~yearInt['year'].between(1880, 2026)).sum()}")
sample(negYear[["_id", "title", "year", "imdbId"]])

Años enteros: min=-2015 max=2016
Años negativos: 90
Positivos fuera de [1880, 2026]: 0


,_id,title,year,imdbId
10049,573a1397f29313caabce622c,Nickelodeon,-1976,74964
5437,573a1398f29313caabce8fc4,Beverly Hills Cop,-1984,86960
12621,573a13d3f29313caabd96e43,People of a Feather,-2011,1929346
15701,573a13c0f29313caabd627c9,Cold in July,-2014,1179031
48,573a1399f29313caabcee89f,Mina Tannenbaum,-1994,110521
7346,573a1397f29313caabce68fa,La Soufrière,-1977,76741
9959,573a13b7f29313caabd4b341,Grave Decisions,-2006,780180
14781,573a13c7f29313caabd74999,The Help,-2011,1454029


**Validación cruzada con duplicados:** si una película con año anómalo tiene otra copia (mismo `imdb.id`) con año
entero válido, podemos comprobar si la regla de rescate da el mismo valor.

In [17]:
def crossCheck(field: str, isAnomalous, rescue) -> pd.DataFrame:
    """Para cada documento anómalo con una copia válida del mismo imdb.id, compara el valor rescatado con el válido."""
    rows = []
    for _, row in moviesDf[moviesDf[field].map(isAnomalous)].iterrows():
        valid = [v for v in moviesDf.loc[moviesDf["imdbId"] == row["imdbId"], field]
                 if not isAnomalous(v) and isinstance(v, int)]
        if valid:
            rows.append({"title": row["title"], "anómalo": repr(row[field]), "copia válida": valid[0],
                         "rescatado": rescue(row[field]), "coincide": rescue(row[field]) == valid[0]})
    return pd.DataFrame(rows)


yearCheck = crossCheck("year", lambda y: not (isinstance(y, int) and y > 0), rescueYear)
print(yearCheck.groupby(yearCheck["anómalo"].map(pattern))["coincide"].agg(["count", "sum"]))
yearCheck.head(10)

             count  sum
anómalo                
"'N'"            8    8
"'Nè'"           8    8
"'circa N'"      8    8
'-N'             3    3
'None'           9    0


,title,anómalo,copia válida,rescatado,coincide
0,Pola X,'circa 1999',1999,1999.0,True
1,End of Days,'circa 1999',1999,1999.0,True
2,Becoming Chaz,None,2011,NaN,False
3,Who Is Harry Nilsson (And Why Is Everybody Talkin' About Him?),None,2010,NaN,False
4,In the Mood for Love,None,2000,NaN,False
5,Divine Trash,'1998è',1998,1998.0,True
6,The International,'2006',2006,2006.0,True
7,Mad As Hell,None,2014,NaN,False
8,Scarlett,'circa 1994',1994,1994.0,True
9,Seventeen Again,None,2000,NaN,False


### 2.3 Duración (`runtime`)

In [18]:
patternTable(moviesDf.loc[moviesDf["runtime"].map(lambda r: not isinstance(r, int)), "runtime"])

,casos,ejemplos
patrón,,
<faltante>,225,<faltante>
<NoneType>,91,None
'N min',91,'100 min' · '136 min' · '114 min' · '88 min'
'N',88,'122' · '137' · '93' · '82'
'Nh Nm',76,'1h 25m' · '2h 4m' · '1h 21m' · '0h 16m'


In [19]:
rtInt = moviesDf[moviesDf["runtime"].map(lambda r: isinstance(r, int))].copy()
rtInt["runtime"] = rtInt["runtime"].astype(int)
print(f"Negativos: {(rtInt['runtime'] < 0).sum()}  ·  Cero: {(rtInt['runtime'] == 0).sum()}")
print("\nDistribución de los positivos por tipo:")
display(rtInt[rtInt["runtime"] > 0].groupby("type")["runtime"].describe(percentiles=[.01, .5, .99]))
sample(rtInt[rtInt["runtime"] < 0][["_id", "title", "runtime", "imdbId"]])

Negativos: 78  ·  Cero: 0

Distribución de los positivos por tipo:


,count,mean,std,min,1%,50%,99%,max
type,,,,,,,,
movie,18989.0,103.338301,24.549330,2.0,29.00,100.0,181.00,551.0
series,204.0,242.691176,186.814409,21.0,23.03,197.5,909.01,1256.0


,_id,title,runtime,imdbId
6985,573a13e8f29313caabdca460,The Nostalgist,-17,3215822
169,573a13a0f29313caabd02a73,Small Time Crooks,-94,196216
7709,573a139af29313caabcef6a8,The Quick and the Dead,-107,114214
3451,573a13b1f29313caabd37444,What Remains of Us,-77,420541
2330,573a1397f29313caabce749e,The Rose,-125,79826
18967,573a13e6f29313caabdc695c,52 Tuesdays,-114,3100636
6683,573a13c2f29313caabd67007,Ein Teil von mir,-88,1251370
1142,573a13b4f29313caabd3fbd9,A Bittersweet Life,-120,456912


¿Qué es "fuera de rango razonable"? Depende del tipo: las `series` tienen la duración total de la temporada,
y los cortos (`Short`) duran pocos minutos. Se cuentan los extremos según distintos umbrales.

In [20]:
def hasGenre(genres, name):
    return isinstance(genres, list) and any(g.strip() == name for g in genres) or (isinstance(genres, str) and name in genres)

pos = rtInt[rtInt["runtime"] > 0].copy()
pos["isShort"] = pos["genres"].map(lambda g: hasGenre(g, "Short"))
print("Largos (> 300 min) por tipo:")
display(pos[pos["runtime"] > 300].groupby("type").size().rename("casos").to_frame())
display(sample(pos[(pos["runtime"] > 300) & (pos["type"] == "movie")][["title", "runtime", "type", "genres"]], 10))
print("Muy cortos (< 15 min): ¿son cortos?")
display(pos[pos["runtime"] < 15].groupby("isShort").size().rename("casos").to_frame())
sample(pos[(pos["runtime"] < 15) & ~pos["isShort"]][["title", "runtime", "genres"]], 10)

Largos (> 300 min) por tipo:


,casos
type,
movie,12
series,45


,title,runtime,type,genres
17908,The Deluge,315,movie,"[Adventure, History, Romance]"
17207,Satantango,450,movie,"[Comedy, Drama]"
742,Hitler: A Film from Germany,442,movie,"[Biography, Drama, History]"
14758,Homeland (Iraq Year Zero),334,movie,[Documentary]
12876,"La Commune (Paris, 1871)",345,movie,"[Drama, History, War]"
4866,The Best of Youth,383,movie,"[Drama, Romance]"
1940,Les vampires,399,movie,"[Action, Adventure, Crime]"
18648,Little Dorrit,357,movie,"[Drama, Romance]"
12553,Heaven's Gate,325,movie,"[Adventure, Drama, History]"
14095,1900,317,movie,[Drama]


Muy cortos (< 15 min): ¿son cortos?


,casos
isShort,
False,6
True,108


,title,runtime,genres
165,Mount Head,10,"[Animation, Comedy, Fantasy]"
1319,Duck Amuck,7,"[Family, Animation, Comedy]"
17789,Three Little Pigs,8,"[Animation, Musical, Family]"
7124,Duck Dodgers in the 24èth Century,7,"[Family, Sci-Fi, Animation]"
11530,Steamboat Willie,8,"[Animation, Family, Comedy]"
8524,The Chubbchubbs!,6,"[Animation, Comedy, Sci-Fi]"


In [21]:
def rescueRuntime(r):
    """Solo para validar: 'Nh Nm' -> minutos, 'N min'/'N' -> N, negativo -> abs."""
    if isinstance(r, int):
        return abs(r)
    if isinstance(r, str):
        hm = re.fullmatch(r"\s*(\d+)h\s*(\d+)m\s*", r)
        if hm:
            return int(hm.group(1)) * 60 + int(hm.group(2))
        num = re.fullmatch(r"\s*(\d+)(\s*min)?\s*", r)
        return int(num.group(1)) if num else None
    return None

rtCheck = crossCheck("runtime", lambda r: not (isinstance(r, int) and r > 0), rescueRuntime)
print(rtCheck.groupby(rtCheck["anómalo"].map(pattern))["coincide"].agg(["count", "sum"]))
rtCheck.head(10)

           count  sum
anómalo              
"'N min'"      4    4
"'N'"          3    3
"'Nh Nm'"      3    3
'-N'           6    6
'None'         3    0


,title,anómalo,copia válida,rescatado,coincide
0,Small Time Crooks,-94,94,94.0,True
1,Underworld: Evolution,-106,106,106.0,True
2,What Remains of Us,-77,77,77.0,True
3,The Ides of March,'101 min',101,101.0,True
4,A Perfect Murder,-107,107,107.0,True
5,Underground: The Julian Assange Story,None,89,NaN,False
6,Malibu's Most Wanted,'86 min',86,86.0,True
7,Bowfinger,'97',97,97.0,True
8,Ronja Robbersdaughter,'2h 6m',126,126.0,True
9,Strangers,None,80,NaN,False


### 2.4 Géneros (`genres`)

In [22]:
genreStr = moviesDf.loc[moviesDf["genres"].map(lambda g: isinstance(g, str)), "genres"]
print("Tipos:", moviesDf["genres"].map(lambda g: type(g).__name__ if g is not MISSING else "<faltante>").value_counts().to_dict())

# Separador = todo lo que no es letra ni guion (Sci-Fi, Film-Noir llevan guion)
separators = Counter(s for g in genreStr for s in re.findall(r"[^A-Za-z\-]+", g.strip()))
print("Separadores encontrados:", {repr(k): v for k, v in separators.items()})
sample(genreStr.to_frame(), 10)

Tipos: {'list': 19421, 'str': 378, '<faltante>': 43}
Separadores encontrados: {"' / '": 210, "'|'": 191, "', '": 172}


,genres
15505,Horror / Mystery
17533,"Adventure, Horror, Thriller"
17104,Drama|History|Romance
7631,"Documentary, Biography"
3225,Drama / Romance
5088,Crime / Thriller
17978,Crime / Drama / Thriller
4477,Drama|Horror|Thriller
19377,"Horror, Romance"
7808,Crime / Thriller / Action


In [23]:
# Vocabulario completo: tokens crudos de listas y de strings (partidos por cualquier separador)
tokens = []
for g in moviesDf["genres"]:
    if isinstance(g, list):
        tokens += [("lista", t) for t in g]
    elif isinstance(g, str):
        tokens += [("string", t) for t in re.split(r"\s*(?:/|\||,)\s*", g)]

vocab = pd.DataFrame(tokens, columns=["origen", "token"])
vocab["canonical"] = vocab["token"].str.strip()
vocab["key"] = vocab["token"].str.lower().str.replace(r"[^a-z]", "", regex=True)  # detecta Sci-Fi vs SciFi vs sci fi

variants = vocab.groupby("key")["token"].agg(lambda s: sorted(set(map(repr, s))))
print(f"Géneros distintos (por clave normalizada): {vocab['key'].nunique()}")
print(f"Elementos de lista con espacios alrededor: {((vocab['origen'] == 'lista') & (vocab['token'] != vocab['canonical'])).sum()}")
(vocab.groupby("canonical").size().rename("ocurrencias").to_frame()
      .join(vocab.groupby("canonical")["key"].first())
      .assign(variantes=lambda d: d["key"].map(variants)).drop(columns="key")
      .sort_values("ocurrencias", ascending=False))

Géneros distintos (por clave normalizada): 25
Elementos de lista con espacios alrededor: 614


,ocurrencias,variantes
canonical,,
Drama,11660,"[' Drama ', 'Drama']"
Comedy,5955,"[' Comedy ', 'Comedy']"
Romance,3241,"[' Romance ', 'Romance']"
Thriller,2424,"[' Thriller ', 'Thriller']"
Crime,2420,"[' Crime ', 'Crime']"
Action,2329,"[' Action ', 'Action']"
Adventure,1915,"[' Adventure ', 'Adventure']"
Documentary,1736,"[' Documentary ', 'Documentary']"
Horror,1612,"[' Horror ', 'Horror']"


### 2.5 Rating de IMDb (`imdb.rating`) y votos (`imdb.votes`)

In [24]:
def ratingCategory(r) -> str:
    if r is MISSING:
        return "faltante"
    if r is None:
        return "null"
    if isinstance(r, str):
        if r.strip() == "":
            return "str vacío"
        return "str numérico" if re.fullmatch(r"-?\d+(\.\d+)?", r.strip()) else "str otro"
    if r < 0:
        return "negativo"
    if r <= 10:
        return "número en [0, 10]"
    return "número en (10, 100]" if r <= 100 else "número > 100"


moviesDf["ratingCat"] = moviesDf["imdbRating"].map(ratingCategory)
moviesDf["ratingCat"].value_counts().rename("casos").to_frame()

,casos
ratingCat,
"número en [0, 10]",19439
"número en (10, 100]",193
str numérico,135
negativo,57
str vacío,18


In [25]:
outOfScale = moviesDf[moviesDf["ratingCat"].isin(["número en (10, 100]", "negativo"])]
print("Valores fuera de [0, 10]:")
outOfScale["imdbRating"].value_counts().rename("casos").to_frame().T

Valores fuera de [0, 10]:


imdbRating,14.5,-1.0,70.0,65.0,66.0,72.0,79.0,64.0,75.0,68.0,...,78.0,47.0,57.0,54.0,50.0,59.0,45.0,40.0,58.0,43.0
casos,61,57,10,9,7,7,7,6,6,6,...,2,1,1,1,1,1,1,1,1,1


El `14.5` se repite demasiado para ser un dato real y no encaja en ninguna escala (14.5/10 = 1.45 es posible,
pero 61 películas con exactamente el mismo valor no). Para distinguir **escala ×10** de **valor centinela** se compara
contra un rating independiente (`metacritic`, escala 0–100): si los ×10 son reales, deberían correlacionar.

In [26]:
metacritic = pd.Series([m.get("metacritic") for m in movies], dtype="float")
num = pd.to_numeric(moviesDf["imdbRating"].map(lambda r: r if isinstance(r, (int, float)) else None), errors="coerce")
compare = pd.DataFrame({"cat": moviesDf["ratingCat"], "rating": num, "metacritic": metacritic})
compare.loc[compare["rating"] > 10, "cat"] = compare["rating"].map(lambda r: "== 14.5" if r == 14.5 else "(10, 100] sin 14.5")
compare = compare.dropna(subset=["metacritic", "rating"])
compare.groupby("cat").apply(lambda g: pd.Series({
    "casos con metacritic": len(g),
    "corr(rating, metacritic)": round(g["rating"].corr(g["metacritic"]), 2) if g["rating"].nunique() > 1 else None,
    "metacritic min–max": f"{g['metacritic'].min():.0f}–{g['metacritic'].max():.0f}",
}), include_groups=False)

,casos con metacritic,"corr(rating, metacritic)",metacritic min–max
cat,,,
"(10, 100] sin 14.5",54,0.7,28–89
== 14.5,29,NaN,29–90
negativo,21,NaN,9–92
"número en [0, 10]",7607,0.7,1–100


In [27]:
for cat in ["str numérico", "str vacío", "negativo"]:
    print(f"— {cat}")
    display(sample(moviesDf[moviesDf["ratingCat"] == cat][["_id", "title", "imdbRating", "imdbVotes"]], 6))
print("— (10, 100] sin 14.5")
sample(moviesDf[(moviesDf["ratingCat"] == "número en (10, 100]") & (num != 14.5)][["_id", "title", "imdbRating", "imdbVotes"]], 6)

— str numérico


,_id,title,imdbRating,imdbVotes
14469,573a1396f29313caabce4a20,"Drive, He Said",5.9,898
10769,573a13dbf29313caabdaf9a0,Interior. Leather Bar.,5.2,1670
15219,573a13d5f29313caabd9a564,Storage 24,4.4,5192
3455,573a1398f29313caabcebb32,Tècknamn Coq Rouge,5.7,782
7265,573a1398f29313caabce8fe8,Breakin',5.6,4156
10372,573a13c5f29313caabd70252,Baseline,5,431


— str vacío


,_id,title,imdbRating,imdbVotes
701,573a13ddf29313caabdb3f4b,The Ghost and the Whale,,
1008,573a13cff29313caabd88f5b,Scouts Guide to the Zombie Apocalypse,,
8708,573a13e3f29313caabdc08a4,Mary Loss of Soul,,
6653,573a13f2f29313caabddd9d3,Clean Hands,,
4950,573a13ddf29313caabdb4f43,Man Down,,
10400,573a13eff29313caabdd87c5,Learning to Ride,,


— negativo


,_id,title,imdbRating,imdbVotes
169,573a13a0f29313caabd02a73,Small Time Crooks,-1.0,29174
1345,573a13aff29313caabd31856,Wedding Crashers,-1.0,262795
10055,573a13c3f29313caabd6b143,Despicable Me,-1.0,339943
5456,573a13d7f29313caabda5036,Nightfall,-1.0,811
11483,573a13c2f29313caabd67487,Je l'aimais,-1.0,981
17693,573a13f2f29313caabdde08a,The Missing,-1.0,"6,566"


— (10, 100] sin 14.5


,_id,title,imdbRating,imdbVotes
8461,573a13b6f29313caabd45b97,Space Chimps,45.0,8338
12275,573a1397f29313caabce6c89,Autumn Sonata,83.0,13656
2225,573a13d5f29313caabd9b839,The Frozen Ground,64.0,40121
4981,573a139ef29313caabcfd66d,Halloweentown,68.0,6311
11249,573a1395f29313caabce1db6,Vivre Sa Vie,81.0,15582
17187,573a13d7f29313caabda1c8b,Elsa & Fred,65.0,1504


In [28]:
print("imdb.votes no enteros:")
patternTable(moviesDf.loc[moviesDf["imdbVotes"].map(lambda v: not isinstance(v, int)), "imdbVotes"])

imdb.votes no enteros:


,casos,ejemplos
patrón,,
"'N,N'",289,"'1,208' · '42,789' · '3,256' · '123,037'"
'N',115,'798' · '142' · '905' · '506'
'',19,''
"'N,N,N'",1,"'1,109,724'"


### 2.6 Títulos con espacios espurios
Se buscan: espacios al inicio, al final, dobles (o tabs/saltos), y caracteres invisibles
(categorías Unicode `Cc`/`Cf` como zero-width space, o espacios no estándar como `\xa0`).

In [29]:
titles = moviesDf["title"]
invisible = titles.map(lambda t: any(unicodedata.category(c) in ("Cc", "Cf") or (unicodedata.category(c) == "Zs" and c != " ") for c in t))
checks = pd.DataFrame({
    "inicio": titles.str.match(r"^\s"),
    "final": titles.str.contains(r"\s$"),
    "dobles/tabs internos": titles.str.strip().str.contains(r"\s{2,}|[\t\n\r]"),
    "invisibles": invisible,
})
print(checks.sum().rename("casos").to_string())
print(f"\nTítulos con al menos un problema: {checks.any(axis=1).sum()}")
sample(moviesDf.loc[checks.any(axis=1), ["_id", "title"]].assign(repr=lambda d: d["title"].map(repr)).drop(columns="title"), 10)

inicio                  201
final                   216
dobles/tabs internos      0
invisibles                0



Títulos con al menos un problema: 417


,_id,repr
7851,573a1396f29313caabce50e8,' The Long Goodbye'
19008,573a13e8f29313caabdc9d89,'The Kingdom of Dreams and Madness '
19555,573a1397f29313caabce6b79,'Despair '
5055,573a13d7f29313caabda1da0,"""The Woman Who Wasn't There """
3730,573a13a6f29313caabd176de,"""Don't Tempt Me """
19656,573a139af29313caabcef164,'Dead Presidents '
18523,573a1394f29313caabce0808,'Sweet Smell of Success '
11091,573a1397f29313caabce8cdc,'Crime and Punishment '
2157,573a13ccf29313caabd82ce1,'Armadillo '
3133,573a13aef29313caabd2c14a,' Guess Who'


### 2.7 Verificación de la regla secundaria de duplicados (decisión A4)

Regla: dos películas con **distinto** `imdb.id` son la misma si coinciden en título normalizado + año rescatado +
duración rescatada **y** en al menos un campo identificatorio **no vacío** (`directors` o `cast`).

Un campo vacío no cuenta como coincidencia: `[] == []` no prueba nada. Va al final de la sección 2 porque usa
`rescueRuntime` (2.3): compara duraciones ya limpias, como lo haría el agente.

In [30]:
def nonEmptyKey(values):
    """Clave comparable de una lista, o None si está vacía o falta (None nunca coincide con nada)."""
    return tuple(sorted(v.strip() for v in values)) if isinstance(values, list) and values else None


def sameMovieEvidence(a: dict, b: dict) -> list[str]:
    """Campos identificatorios no vacíos en los que coinciden dos documentos."""
    evidence = []
    for field in ("directors", "cast"):
        ka, kb = nonEmptyKey(a.get(field)), nonEmptyKey(b.get(field))
        if ka is not None and ka == kb:
            evidence.append(field)
    return evidence


rows = []
for (title, year), group in dupB2.groupby(["titleNorm", "yearRescued"]):
    ids = list(dict.fromkeys(group["imdbId"]))
    for i, idA in enumerate(ids):
        for idB in ids[i + 1:]:
            a, b = docsById[idA][0], docsById[idB][0]
            rtA, rtB = rescueRuntime(a.get("runtime")), rescueRuntime(b.get("runtime"))
            evidence = sameMovieEvidence(a, b)
            rows.append({
                "título": title, "año": int(year), "imdb.ids": (idA, idB),
                "misma duración": rtA is not None and rtA == rtB,
                "evidencia": ", ".join(evidence) or "—",
                "regla A4": rtA is not None and rtA == rtB and bool(evidence),
            })

ruleA4 = pd.DataFrame(rows).sort_values("regla A4", ascending=False)
print(f"Pares título+año con distinto imdb.id: {len(ruleA4)}")
print(f"Pares que la regla A4 considera duplicados: {ruleA4['regla A4'].sum()}")
ruleA4

Pares título+año con distinto imdb.id: 25
Pares que la regla A4 considera duplicados: 4


,título,año,imdb.ids,misma duración,evidencia,regla A4
22,the kingdom,1994,"(108906, 127392)",True,cast,True
7,confessions of a dangerous mind,2002,"(270288, 290538)",True,"directors, cast",True
9,divide and conquer,1943,"(182989, 173769)",True,"directors, cast",True
11,eros,2004,"(377059, 343663)",True,"directors, cast",True
0,1,2013,"(2518788, 2395385)",False,—,False
14,grace,2014,"(2963344, 3068192)",False,—,False
23,the suspect,2013,"(3404158, 2073121)",False,—,False
21,the girl,2012,"(1127883, 2132485)",False,—,False
20,the castle,1997,"(118826, 120075)",False,—,False
19,soldier,1998,"(120157, 211634)",False,—,False


## 3. Anomalías conocidas — `reviews` (Paso 3b)

Mismo enfoque que en `movies`: DataFrame plano sin convertir tipos.

In [31]:
from datetime import datetime, timezone

reviewsDf = pd.DataFrame({
    "_id": [str(r["_id"]) for r in reviews],
    "movieId": [r["movie_id"] for r in reviews],
    "userId": [r.get("user_id", MISSING) for r in reviews],
    "userName": [r.get("user_name", MISSING) for r in reviews],
    "userEmail": [r.get("user_email", MISSING) for r in reviews],
    "rating": [r.get("rating", MISSING) for r in reviews],
    "comment": [r.get("comment", MISSING) for r in reviews],
    "date": [r.get("date", MISSING) for r in reviews],
}, dtype=object)
reviewsDf.head(3)

,_id,movieId,userId,userName,userEmail,rating,comment,date
0,6af3753e68313c9a2da138a5,573a1397f29313caabce8475,usr_0937,Leo Roberts,leo.roberts551@proton.me,4,A very compelling watch with standout lead performances. A well-earned 4 stars.,2024-03-10 00:01:43
1,6af3753e68313c9a2da0c6d6,573a13a5f29313caabd13b4b,usr_0177,Casey Adams,casey.adams980@yahoo.com,3,"Not bad, but ultimately didn't leave a lasting impression. A few funny and well-timed jokes, but the overall plot is...",2025-09-01 06:57:51
2,6af3753e68313c9a2da13941,573a1399f29313caabcee2fa,usr_0942,Logan Lopez,logan.lopez152@proton.me,3,"Has enough entertainment value to justify a casual watch. Good effort from the crew, though the third act falls into...",2024-05-06 15:55:09


### 3.1 Comentarios basura o spam

**Primero: ¿repetido = spam?** Las reseñas son sintéticas, armadas con plantillas. Si muchos comentarios legítimos
se repiten, la frecuencia no sirve como criterio de spam.

In [32]:
comments = reviewsDf["comment"]
sentences = Counter(s.strip() for c in comments for s in re.split(r"(?<=[.!?])\s+", c) if s.strip())
print(f"Comentarios: {len(comments)} · distintos: {comments.str.strip().str.lower().nunique()}")
print(f"Oraciones distintas en todo el dataset: {len(sentences)}")
print("\nComentarios exactos más repetidos:")
comments.value_counts().head(8).to_frame("veces")

Comentarios: 61488 · distintos: 1669
Oraciones distintas en todo el dataset: 107

Comentarios exactos más repetidos:


,veces
comment,
Exceeded my expectations on multiple levels. The acting is top-notch and the production values are noticeably high.,205
Really solid piece of entertainment with a lot of heart. The acting is top-notch and the production values are noticeably high.,204
Impressive direction and great creative vision throughout. The supporting cast brings genuine charm and elevates the central conflict.,201
"Exceeded my expectations on multiple levels. The story takes some predictable turns, but the emotional beats land effectively.",199
A strong entry with plenty of memorable moments. Great visual style and smart editing keep the energy moving nicely.,192
Definitely worth watching on a weekend night. Great visual style and smart editing keep the energy moving nicely.,191
Exceeded my expectations on multiple levels. Great visual style and smart editing keep the energy moving nicely.,190
Really solid piece of entertainment with a lot of heart. Great visual style and smart editing keep the energy moving nicely.,189


Los comentarios más repetidos son plantillas legítimas (~1.700 distintos armados con ~100 oraciones),
y el string vacío aparece entre ellos. **La repetición no distingue spam.**

**Segundo: distribución de largo en palabras.** Si hay un "hueco" entre los comentarios basura y los reales,
el umbral sale de los datos y no es arbitrario. Se quita antes el prefijo `"Update: Changed my mind after rewatching."`
(ver 3.4), que se antepone a algunos comentarios.

In [33]:
UPDATE_PREFIX = "Update: Changed my mind after rewatching."

def stripUpdatePrefix(text: str) -> str:
    return text[len(UPDATE_PREFIX):].strip() if text.startswith(UPDATE_PREFIX) else text.strip()

body = comments.map(stripUpdatePrefix)
wordCount = body.str.split().str.len()
wordCount.value_counts().sort_index().head(12).to_frame("comentarios").T

comment,0,1,2,9,10,11,12,13,14,15,16,17
comentarios,196,1293,94,247,419,424,1177,1463,1691,1645,2043,2501


Hay un hueco claro: nada entre 3 y 8 palabras. Todo lo de 0–2 palabras es candidato a basura.

In [34]:
CATEDRA_LIST = {"...", "", "none", "test"}

shortBodies = body[wordCount < 3]
garbageTable = shortBodies.value_counts().rename("casos").to_frame()
garbageTable["sin letras"] = [not re.search(r"[A-Za-z]", v) for v in garbageTable.index]
garbageTable["en lista de la cátedra"] = [v.strip().lower() in CATEDRA_LIST for v in garbageTable.index]
garbageTable.index = garbageTable.index.map(repr)
print(f"Comentarios de 0–2 palabras: {len(shortBodies)}")
garbageTable

Comentarios de 0–2 palabras: 1583


,casos,sin letras,en lista de la cátedra
comment,,,
'',196,True,True
'...',158,True,True
'???',97,True,False
'1234',97,True,False
'none',96,False,True
'ok',94,False,False
'no comment',94,False,False
'idk',89,False,False
'nice',88,False,False


**Caso borde:** `'nice'`, `'good'`, `'ok'` no son ruido puro, porque expresan algo de sentimiento, pero no alcanzan
para inferir un rating de 1 a 5 (¿`'good'` es 3, 4 o 5?). Se los clasifica aparte como *bajo contenido*.

In [35]:
LOW_CONTENT = {"nice", "good", "ok"}

def commentQuality(text) -> str:
    """'útil', 'bajo contenido' o 'basura'. Se evalúa sin el prefijo de Update."""
    if not isinstance(text, str):
        return "basura"
    b = stripUpdatePrefix(text)
    if len(b.split()) >= 3:
        return "útil"
    return "bajo contenido" if b.lower() in LOW_CONTENT else "basura"

reviewsDf["commentQuality"] = reviewsDf["comment"].map(commentQuality)
print(reviewsDf["commentQuality"].value_counts().to_string())
print("\nBasura detrás del prefijo Update:")
reviewsDf.loc[reviewsDf["comment"].str.startswith(UPDATE_PREFIX) & (reviewsDf["commentQuality"] != "útil"), "comment"].map(repr).value_counts().to_frame("casos")

commentQuality
útil              59905
basura             1315
bajo contenido      268

Basura detrás del prefijo Update:


,casos
comment,
'Update: Changed my mind after rewatching. ...',3
'Update: Changed my mind after rewatching. nice',2
'Update: Changed my mind after rewatching. movie',1
'Update: Changed my mind after rewatching. asdf',1


### 3.2 Fechas (`date`)

In [36]:
patternTable(reviewsDf.loc[reviewsDf["date"].map(lambda d: not isinstance(d, datetime)), "date"])

,casos,ejemplos
patrón,,
'yesterday',197,'yesterday'
'circa N',184,'circa 2023'
'N/N/N',179,'06/04/2024' · '12/09/2023' · '14/04/2026' · '01/09/2024'
<int>,177,1707057516000 · 1720527134000 · 1750458205000 · 1767931144000
'N-N-N',173,'2026-02-18' · '2026-01-30' · '2023-10-11' · '2025-11-09'


**Enteros: ¿segundos o milisegundos epoch?** Se convierten de las dos formas y se compara con el rango de las fechas válidas.

In [37]:
validDates = reviewsDf["date"][reviewsDf["date"].map(lambda d: isinstance(d, datetime))]
intDates = reviewsDf["date"][reviewsDf["date"].map(lambda d: isinstance(d, int))].astype("int64")
print(f"Rango de fechas válidas:        {validDates.min()}  →  {validDates.max()}")
print(f"Enteros como milisegundos:      {pd.to_datetime(intDates.min(), unit='ms')}  →  {pd.to_datetime(intDates.max(), unit='ms')}")
print(f"Enteros como segundos:          {pd.to_datetime(intDates.min(), unit='s')}  →  {pd.to_datetime(intDates.max(), unit='s')}")

Rango de fechas válidas:        2023-07-19 00:02:12  →  2027-04-11 15:22:05
Enteros como milisegundos:      2023-07-27 23:35:07  →  2026-10-23 11:19:35
Enteros como segundos:          55539-11-04 17:16:40  →  58780-02-15 22:23:20


**Formato `N/N/N`: ¿dd/mm o mm/dd?** Decisión B2: gana la convención mayoritaria de cada fuente, y las fechas no ambiguas
(algún número > 12) son las que la revelan.

In [38]:
slashDates = reviewsDf["date"][reviewsDf["date"].map(lambda d: isinstance(d, str) and "/" in d)]

def slashKind(s: str) -> str:
    first, second, _ = map(int, s.split("/"))
    if first > 12:
        return "primero > 12 → dd/mm"
    if second > 12:
        return "segundo > 12 → mm/dd"
    return "ambigua (ambos ≤ 12)"

slashDates.map(slashKind).value_counts().to_frame("casos")

,casos
date,
primero > 12 → dd/mm,105
ambigua (ambos ≤ 12),74


In [39]:
isoDates = reviewsDf["date"][reviewsDf["date"].map(lambda d: isinstance(d, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", d) is not None)]
print(f"Formato N-N-N: {len(isoDates)}, todos con año de 4 dígitos al principio → ISO yyyy-mm-dd.",
      f"Mes > 12: {(isoDates.str[5:7].astype(int) > 12).sum()}")

# Fechas en el futuro respecto de la creación del dataset (timestamp de los ObjectId de reviews)
datasetCreated = min(r["_id"].generation_time for r in reviews).replace(tzinfo=None)
future = validDates[validDates > datasetCreated]
print(f"\nDataset generado (según ObjectId): {datasetCreated}")
print(f"Fechas válidas posteriores a esa fecha: {len(future)} (máx. {future.max()})")

releasedById = {m["_id"]: m.get("released") for m in movies}
beforeRelease = sum(1 for mid, d in zip(reviewsDf["movieId"], reviewsDf["date"])
                    if isinstance(d, datetime) and isinstance(releasedById.get(mid), datetime) and d < releasedById[mid])
print(f"Reseñas fechadas antes del estreno de la película: {beforeRelease}")

Formato N-N-N: 173, todos con año de 4 dígitos al principio → ISO yyyy-mm-dd. Mes > 12: 0



Dataset generado (según ObjectId): 2026-11-10 19:49:18
Fechas válidas posteriores a esa fecha: 30 (máx. 2027-04-11 15:22:05)


Reseñas fechadas antes del estreno de la película: 0


### 3.3 Reseñas huérfanas (`movie_id` inexistente)

In [40]:
movieIds = {m["_id"] for m in movies}
orphans = reviewsDf[~reviewsDf["movieId"].isin(movieIds)]
print(f"Reseñas huérfanas: {len(orphans)} · movie_id distintos: {orphans['movieId'].nunique()}")

# ¿De dónde salen esos ids? Se compara su prefijo (timestamp del ObjectId) con los de movies y reviews
prefix = lambda oid: str(oid)[:8]
print("Prefijos de movie_id huérfanos:", Counter(map(prefix, orphans["movieId"])).most_common(3))
print("Prefijos de _id de movies:     ", Counter(prefix(m["_id"]) for m in movies).most_common(3))
print("Prefijos de _id de reviews:    ", Counter(prefix(r["_id"]) for r in reviews).most_common(3))
sample(orphans[["_id", "movieId", "userId", "rating"]])

Reseñas huérfanas: 350 · movie_id distintos: 50
Prefijos de movie_id huérfanos: [('6af3753e', 350)]
Prefijos de _id de movies:      [('573a139a', 1303), ('573a1398', 1261), ('573a1399', 1107)]


Prefijos de _id de reviews:     [('6af3753e', 53080), ('6af3753f', 8408)]


,_id,movieId,userId,rating
27455,6af3753f68313c9a2da18b67,6af3753e68313c9a2da09cf9,usr_0817,3
59190,6af3753f68313c9a2da18b02,6af3753e68313c9a2da09d17,usr_0118,1
54793,6af3753f68313c9a2da18a81,6af3753e68313c9a2da09d20,usr_1007,4
41036,6af3753f68313c9a2da18b8e,6af3753e68313c9a2da09d09,usr_0658,5
27417,6af3753f68313c9a2da18aad,6af3753e68313c9a2da09d21,usr_1171,2
48446,6af3753f68313c9a2da18aff,6af3753e68313c9a2da09d00,usr_1870,1
53110,6af3753f68313c9a2da18b49,6af3753e68313c9a2da09d0a,usr_0511,2
39584,6af3753f68313c9a2da18a7d,6af3753e68313c9a2da09d07,usr_1360,3


### 3.4 Votos duplicados (mismo usuario sobre la misma película)
Se excluyen los usuarios anónimos (`''` y `usr_0000`, ver 3.6): no se puede saber si son la misma persona.

In [41]:
ANONYMOUS_IDS = {"", "usr_0000"}
identified = reviewsDf[~reviewsDf["userId"].isin(ANONYMOUS_IDS)]
dupVotes = identified[identified.duplicated(["userId", "movieId"], keep=False)].copy()
pairs = dupVotes.groupby(["userId", "movieId"])
print(f"Pares duplicados: {pairs.ngroups} · tamaños de grupo: {pairs.size().value_counts().to_dict()}")

def differs(field):
    return pairs[field].agg(lambda s: s.map(json_util.dumps).nunique() > 1)

diffTable = pd.DataFrame({f: differs(f) for f in ["rating", "comment", "date"]})
diffTable.value_counts().rename("pares").to_frame()

Pares duplicados: 448 · tamaños de grupo: {2: 448}


pares
rating comment date        
False  False   True     221
True   True    True     209
False  True    True      10
       False   False      5
True   True    False      3

Dos tipos de duplicado: **copias** (mismo rating y comentario, cambia solo la fecha) y **actualizaciones**
(el comentario nuevo empieza con `"Update: Changed my mind after rewatching."`). Para la decisión B1 (gana el más reciente)
se verifica que la actualización sea efectivamente la más reciente.

In [42]:
def toDateOrNone(d):
    return d if isinstance(d, datetime) else None

rows = []
for (user, movie), g in pairs:
    dates = g["date"].map(toDateOrNone)
    hasUpdate = g["comment"].str.startswith(UPDATE_PREFIX).any()
    latestIsUpdate = None
    if dates.notna().all():
        latestIsUpdate = g.loc[dates.astype("datetime64[ns]").idxmax(), "comment"].startswith(UPDATE_PREFIX)
    rows.append({"tipo": "actualización" if hasUpdate else "copia", "fechas comparables": dates.notna().all(),
                 "la más reciente es el Update": latestIsUpdate})

dupKinds = pd.DataFrame(rows)
display(dupKinds.groupby(["tipo", "fechas comparables"]).size().rename("pares").to_frame())
print("En actualizaciones con fechas comparables, ¿la más reciente es el Update?")
print(dupKinds[(dupKinds["tipo"] == "actualización") & dupKinds["fechas comparables"]]["la más reciente es el Update"].value_counts().to_string())

pares
tipo          fechas comparables       
actualización False                   3
              True                  219
copia         False                   5
              True                  221

En actualizaciones con fechas comparables, ¿la más reciente es el Update?
la más reciente es el Update
True    219


In [43]:
# Ejemplos: 2 pares de cada tipo
updateKeys = dupVotes[dupVotes["comment"].str.startswith(UPDATE_PREFIX)][["userId", "movieId"]].drop_duplicates().head(2)
copyKeys = (dupVotes.groupby(["userId", "movieId"]).filter(lambda g: not g["comment"].str.startswith(UPDATE_PREFIX).any())
            [["userId", "movieId"]].drop_duplicates().head(2))
for keys in (updateKeys, copyKeys):
    for _, k in keys.iterrows():
        display(dupVotes[(dupVotes["userId"] == k["userId"]) & (dupVotes["movieId"] == k["movieId"])]
                [["userId", "rating", "date", "comment"]].assign(comment=lambda d: d["comment"].str[:90]))

,userId,rating,date,comment
296,usr_0153,5,2025-07-14 16:45:24,Update: Changed my mind after rewatching. A very compelling watch with standout lead perfo
7956,usr_0153,4,2025-05-31 16:45:24,A very compelling watch with standout lead performances. While the middle section drags sl


,userId,rating,date,comment
708,usr_0297,5,2026-12-05 07:39:29,Update: Changed my mind after rewatching. A very compelling watch with standout lead perfo
32048,usr_0297,4,2026-07-11 07:39:29,A very compelling watch with standout lead performances. The acting is top-notch and the p


,userId,rating,date,comment
56,usr_0262,4,2023-12-19 19:58:12,Exceeded my expectations on multiple levels. Easily recommended for any movie enthusiast.
5868,usr_0262,4,2023-09-05 19:58:12,Exceeded my expectations on multiple levels. Easily recommended for any movie enthusiast.


,userId,rating,date,comment
143,usr_0305,3,2024-03-13 22:24:16,"Competently made, yet somewhat forgettable. The main actors do their best with a script th"
39491,usr_0305,3,2023-11-10 22:24:16,"Competently made, yet somewhat forgettable. The main actors do their best with a script th"


### 3.5 `rating: null` (candidatas a imputación con LLM)
Se cruza con la calidad del comentario (3.1): solo las de texto útil se pueden imputar. Decisión B3: primero ver si existe el caso
"null + basura".

In [44]:
nullRating = reviewsDf[reviewsDf["rating"].map(lambda r: r is None)]
print(f"rating null: {len(nullRating)}")
print(nullRating["commentQuality"].value_counts().to_string())
sample(nullRating[["_id", "userId", "comment"]].assign(comment=lambda d: d["comment"].str[:100]), 6)

rating null: 928
commentQuality
útil    928


,_id,userId,comment
51353,6af3753e68313c9a2da140f9,usr_0997,"A very compelling watch with standout lead performances. While the middle section drags slightly, th"
45564,6af3753e68313c9a2da128bd,usr_0821,A very compelling watch with standout lead performances. The acting is top-notch and the production
1605,6af3753f68313c9a2da187be,usr_1965,An enjoyable premise that plays it a bit too safe. A solid 3-star effort.
27425,6af3753e68313c9a2da11c90,usr_0731,"Definitely worth watching on a weekend night. While the middle section drags slightly, the strong cl"
12268,6af3753e68313c9a2da0c062,usr_0151,Struggles to find its tone and ends up feeling disjointed. The choices made by the protagonist make
25252,6af3753e68313c9a2da0dd46,usr_0273,"Total waste of time and talent. Cringe-worthy dialogue, amateurish editing, and zero narrative coher"


### 3.6 `user_id`

In [45]:
print(patternTable(reviewsDf["userId"]).to_string())
validUserNum = reviewsDf["userId"].map(lambda u: int(u[4:]) if isinstance(u, str) and re.fullmatch(r"usr_\d{4}", u) else None)
print(f"\nFuera del rango 0001–2000: {((validUserNum < 1) | (validUserNum > 2000)).sum()}",
      f"→ valores: {sorted(reviewsDf.loc[(validUserNum < 1) | (validUserNum > 2000), 'userId'].unique())}")

         casos                                           ejemplos
patrón                                                           
'usr_N'  61260  'usr_0937' · 'usr_0177' · 'usr_0942' · 'usr_0926'
''         228                                                 ''



Fuera del rango 0001–2000: 231 → valores: ['usr_0000']


In [46]:
anon = reviewsDf[reviewsDf["userId"].isin(ANONYMOUS_IDS)]
print("Usuarios anónimos: user_id, user_name y email")
display(anon.groupby(["userId", "userName"]).agg(casos=("_id", "size"), emailsNoVacios=("userEmail", lambda s: (s.str.strip() != "").sum())))

# ¿Se pueden recuperar? Solo si email o nombre identifican a un único usuario
known = reviewsDf[~reviewsDf["userId"].isin(ANONYMOUS_IDS)]
idsPerEmail = known.groupby("userEmail")["userId"].nunique()
idsPerName = known.groupby("userName")["userId"].nunique()
print(f"Emails con más de un user_id: {(idsPerEmail > 1).sum()} de {len(idsPerEmail)} → el email identifica unívocamente")
print(f"Nombres con más de un user_id: {(idsPerName > 1).sum()} de {len(idsPerName)} → el nombre NO identifica")
print(f"Anónimos con email no vacío (recuperables por email): {(anon['userEmail'].str.strip() != '').sum()}")

Usuarios anónimos: user_id, user_name y email


,,casos,emailsNoVacios
userId,userName,,
,Anonymous,228,0
usr_0000,Anonymous,231,0


Emails con más de un user_id: 0 de 2000 → el email identifica unívocamente
Nombres con más de un user_id: 432 de 1444 → el nombre NO identifica
Anónimos con email no vacío (recuperables por email): 0


### 3.7 Ratings fuera de 1–5 o no enteros

In [47]:
def reviewRatingCategory(r) -> str:
    if r is None:
        return "null"
    if isinstance(r, str):
        return "str entero 1–5" if r.strip() in {"1", "2", "3", "4", "5"} else "str otro"
    if isinstance(r, float) and not r.is_integer():
        return "decimal"
    return "entero 1–5" if 1 <= r <= 5 else f"entero fuera de rango ({int(r)})"

reviewsDf["ratingCat"] = reviewsDf["rating"].map(reviewRatingCategory)
reviewsDf["ratingCat"].value_counts().to_frame("casos")

,casos
ratingCat,
entero 1–5,58476
null,928
str entero 1–5,873
entero fuera de rango (10),228
entero fuera de rango (-1),212
entero fuera de rango (999),207
entero fuera de rango (100),196
entero fuera de rango (0),189
entero fuera de rango (6),179


**¿Escala distinta o ruido?** Muchos comentarios dicen explícitamente *"A well-earned 4 stars"*. Si `10` fuera una escala
1–10, esas reseñas deberían decir 5 estrellas; si `100` fuera 0–100, lo mismo. Se cruza el rating guardado con las estrellas del texto.

In [48]:
STARS = re.compile(r"(\d+)[- ]stars?", re.IGNORECASE)
reviewsDf["textStars"] = reviewsDf["comment"].map(lambda c: int(m.group(1)) if isinstance(c, str) and (m := STARS.search(c)) else None)
withStars = reviewsDf.dropna(subset=["textStars"])
print(f"Reseñas que mencionan 'N stars': {len(withStars)}")
pd.crosstab(withStars["rating"].map(repr), withStars["textStars"].astype(int), margins=True, margins_name="total")

Reseñas que mencionan 'N stars': 6648


textStars,1,3,4,total
rating,,,,
'1',2,0,0,2
'3',0,48,0,48
'4',0,0,46,46
-1,0,9,14,23
0,1,12,11,24
1,142,3,0,145
10,1,10,14,25
100,1,9,14,24
2,0,5,6,11


In [49]:
validWithStars = withStars[withStars["ratingCat"] == "entero 1–5"]
agree = (validWithStars["rating"].astype(int) == validWithStars["textStars"]).mean()
print(f"Concordancia rating válido vs. estrellas del texto: {agree:.1%} ({len(validWithStars)} reseñas)")
sample(withStars[withStars["ratingCat"].str.startswith("entero fuera")][["_id", "rating", "textStars", "comment"]]
       .assign(comment=lambda d: d["comment"].str[:90]), 8)

Concordancia rating válido vs. estrellas del texto: 99.6% (6307 reseñas)


,_id,rating,textStars,comment
37896,6af3753e68313c9a2da0ede9,100,4.0,A strong entry with plenty of memorable moments. A well-earned 4 stars.
43851,6af3753f68313c9a2da16e04,6,4.0,A very compelling watch with standout lead performances. A well-earned 4 stars.
5397,6af3753e68313c9a2da0b1e2,100,4.0,Impressive direction and great creative vision throughout. Great visual style and smart ed
10573,6af3753e68313c9a2da0b3ad,6,4.0,"A strong entry with plenty of memorable moments. While the middle section drags slightly,"
28237,6af3753f68313c9a2da16dbc,100,4.0,A thoroughly engaging and well-crafted movie. Great visual style and smart editing keep th
29966,6af3753e68313c9a2da1588c,999,3.0,A decent flick if you have a couple of hours to kill. A solid 3-star effort.
29638,6af3753e68313c9a2da0ead9,10,3.0,"Competently made, yet somewhat forgettable. Visually appealing, but the emotional core is"
49601,6af3753e68313c9a2da0aa9d,10,4.0,Definitely worth watching on a weekend night. Great visual style and smart editing keep th


## 4. Anomalías no mencionadas (Paso 4)

Anomalías que la consigna no lista pero aparecen en los datos.

### 4.1 Caracteres perdidos (`è`) en el texto de `movies`

Nombres como `'Franèois Truffaut'` o `'Bjèrk'` sugieren que los caracteres no ASCII se perdieron. Para ver si es un
error puntual o sistemático: en qué campos aparece la `è`, entre qué caracteres, y cuántos caracteres no ASCII hay
en total en cada colección. Usa `walk()` (sección 1), así que también recorre subcampos y elementos de listas.

In [50]:
def charKind(c: str) -> str:
    if c in "^$ ":
        return c
    return "letra" if c.isalpha() else "dígito" if c.isdigit() else repr(c)


def strLeaves(doc: dict):
    """(path, valor) de cada string del documento, incluidos subcampos y elementos de listas."""
    pairs = []
    walk(doc, "", pairs)
    return [(p, v) for p, v in pairs if isinstance(v, str)]


for name, docs in [("movies", movies), ("reviews", reviews)]:
    nonAscii = Counter(ch for d in docs for _, s in strLeaves(d) for ch in s if ord(ch) > 127)
    print(f"{name}: caracteres no ASCII más frecuentes: {nonAscii.most_common(10)}")

eRows = defaultdict(lambda: {"apariciones": 0, "contextos": Counter(), "ejemplos": []})
for m in movies:
    for p, s in strLeaves(m):
        for i, ch in enumerate(s):
            if ch != "è":
                continue
            row = eRows[p]
            row["apariciones"] += 1
            before = s[i - 1] if i > 0 else "^"
            after = s[i + 1] if i + 1 < len(s) else "$"
            row["contextos"][f"{charKind(before)} è {charKind(after)}"] += 1
        if "è" in s and len(eRows[p]["ejemplos"]) < 4:
            eRows[p]["ejemplos"].append(s[:40])

moviesWithE = sum(any("è" in s for _, s in strLeaves(m)) for m in movies)
print(f"\nPelículas con al menos una 'è': {moviesWithE} de {len(movies)} ({moviesWithE / len(movies):.0%})")
pd.DataFrame([
    {"campo": p, "apariciones": r["apariciones"],
     "contextos más comunes": ", ".join(f"{c}: {n}" for c, n in r["contextos"].most_common(3)),
     "ejemplos": " | ".join(repr(e) for e in r["ejemplos"])}
    for p, r in eRows.items()
]).set_index("campo").sort_values("apariciones", ascending=False)

movies: caracteres no ASCII más frecuentes: [('è', 13250), ('\xa0', 7), ('é', 4), ('ä', 4), ('¨', 2), ('í', 2), ('ç', 1), ('ñ', 1)]


reviews: caracteres no ASCII más frecuentes: []



Películas con al menos una 'è': 4933 de 19842 (25%)


,apariciones,contextos más comunes,ejemplos
campo,,,
cast[],5548,"letra è letra: 4252, letra è $: 550, letra è : 513",'Sèren Sètter-Lassen' | 'Gèrard Depardieu' | 'Andrèa Ferrèol' | 'Tran Nu Yèn-Khè'
writers[],3126,"letra è letra: 2328, letra è : 531, letra è $: 101",'Pèr Lagerkvist (novel)' | 'Franèois Truffaut (scenario)' | 'Franèois Truffaut (dialogue)' | 'Sebastièn Silva'
fullplot,1860,"letra è letra: 1281, letra è : 296, letra è ',': 71","'While attending a medical conference in ' | 'The infamous writer, the Marquis de Sade' | ""German village Igelheim's..."
directors[],1571,"letra è letra: 1176, letra è $: 156, letra è : 132",'Franèois Truffaut' | 'Thorbjèrn Christoffersen' | 'Sebastièn Silva' | 'Jens èstberg'
plot,545,"letra è letra: 373, letra è : 85, letra è ',': 32","'In Fèlix and Meira, an unusual romance b' | 'Fifteen-year-old Beni falls in love with' | ""A man, upon entering his ..."
title,292,"letra è letra: 203, letra è : 27, letra è $: 26","'The Brèken' | '8 è Women' | 'Lèon Morin, Priest' | 'Fègi Is a Bastard'"
year,127,"dígito è $: 123, dígito è dígito: 4",'1981è' | '2004è' | '1993è' | '2001è'
tomatoes.consensus,113,"letra è letra: 98, letra è : 9, letra è '.': 4","'From its brilliant cinematography to its' | 'Clichèd and unoriginal, Paranoia is a mi' | 'Philadelphia indulges in ..."
tomatoes.production,66,"letra è letra: 52, letra è : 10, letra è $: 3",'Pathè' | 'Pathè Contemporary Films' | 'ARP Sèlection' | 'Sèville Pictures'


Un mismo nombre (ej. `'Gèrard Depardieu'`) se repite en muchas películas. Para dimensionar una eventual corrección
importa cuántos textos **distintos** hay: es lo que habría que corregir si cada uno se corrige una sola vez y se
reutiliza el resultado.

In [51]:
eValues = defaultdict(list)
for m in movies:
    for p, s in strLeaves(m):
        if "è" in s:
            eValues[p].append(s)

eDistinct = pd.DataFrame(
    {"valores con è": {p: len(v) for p, v in eValues.items()},
     "distintos": {p: len(set(v)) for p, v in eValues.items()}}
).sort_values("valores con è", ascending=False)
eDistinct.loc["TOTAL"] = eDistinct.sum()
eDistinct

,valores con è,distintos
cast[],4718,2521
writers[],2559,1844
directors[],1283,600
fullplot,972,934
plot,432,417
title,255,241
year,127,47
tomatoes.consensus,94,90
tomatoes.production,60,38
countries[],1,1


### 4.2 `imdb.votes` no enteros (decisión A14)

La 2.5 mostró las formas: strings con número (`'1,208'`, `'798'`) y strings vacíos. Acá se comprueba lo que
la regla A14 da por supuesto:
1. ¿Todos los strings con número se convierten sin ambigüedad?
2. Los vacíos, ¿son películas sin datos de IMDb (también sin rating) o se perdió solo el voto?
3. Después de convertir, ¿el rango es razonable (sin ceros ni negativos)?
4. En las películas duplicadas, ¿algún vacío se puede recuperar de otra copia?

In [52]:
VOTES_NUMBER = re.compile(r"\d{1,3}(,\d{3})+|\d+")  # '798', '1,208', '1,109,724'


def parseVotes(v):
    """Regla A14: int se queda; string con número -> int (sin comas); cualquier otra cosa -> None."""
    if isinstance(v, int):
        return v
    if isinstance(v, str) and VOTES_NUMBER.fullmatch(v):
        return int(v.replace(",", ""))
    return None


moviesDf["votesClean"] = moviesDf["imdbVotes"].map(parseVotes)
isStr = moviesDf["imdbVotes"].map(lambda v: isinstance(v, str))
strNumber = moviesDf[isStr & (moviesDf["imdbVotes"] != "")]
strEmpty = moviesDf[moviesDf["imdbVotes"] == ""]
clean = moviesDf["votesClean"].dropna()

# 4. Duplicados: ¿las copias difieren en votos? ¿algún vacío tiene valor en otra copia?
votesById = moviesDf.groupby("imdbId")["votesClean"].agg(list)
votesById = votesById[votesById.map(len) > 1]
differ = votesById.map(lambda vs: len({v for v in vs if v is not None and v == v}) > 1).sum()
recoverable = votesById.map(lambda vs: any(v is None or v != v for v in vs)
                            and any(v is not None and v == v for v in vs)).sum()

pd.DataFrame([
    ("strings con número", len(strNumber)),
    ("  ... que no se pudieron convertir", strNumber["votesClean"].isna().sum()),
    ("strings vacíos ('')", len(strEmpty)),
    ("  ... que además tienen imdb.rating ''", (strEmpty["imdbRating"] == "").sum()),
    ("enteros después de convertir", len(clean)),
    ("  mínimo", int(clean.min())),
    ("  máximo", int(clean.max())),
    ("  ceros o negativos", (clean <= 0).sum()),
    ("grupos duplicados (imdb.id) donde los votos difieren", differ),
    ("grupos duplicados con un vacío recuperable de otra copia", recoverable),
], columns=["chequeo", "valor"]).set_index("chequeo")

,valor
chequeo,
strings con número,405
... que no se pudieron convertir,0
strings vacíos (''),19
... que además tienen imdb.rating '',18
enteros después de convertir,19823
mínimo,6
máximo,1521105
ceros o negativos,0
grupos duplicados (imdb.id) donde los votos difieren,555


### 4.3 `num_mflix_comments` vs. reseñas reales

`num_mflix_comments` es un contador **guardado** en cada película: dice cuántas reseñas tiene. Pero las reseñas
están en `reviews`, así que el número real se puede contar. Si el contador no coincide con el conteo, el campo miente.

Se cuentan las reseñas crudas por `movie_id` (sin limpiar: incluye anónimas, duplicadas, etc.).

In [53]:
reviewsPerMovie = reviewsDf["movieId"].value_counts()

counterDf = pd.DataFrame({
    "title": [m["title"] for m in movies],
    "contador": [m.get("num_mflix_comments", MISSING) for m in movies],
    "reales": [int(reviewsPerMovie.get(m["_id"], 0)) for m in movies],
}, dtype=object)


def counterCategory(row) -> str:
    if row["contador"] is MISSING:
        return "sin campo, con reseñas" if row["reales"] > 0 else "sin campo, sin reseñas"
    if row["contador"] == row["reales"]:
        return "coincide"
    return "contador mayor" if row["contador"] > row["reales"] else "contador menor"


counterDf["categoría"] = counterDf.apply(counterCategory, axis=1)
print(counterDf["categoría"].value_counts().to_string())

withField = counterDf[counterDf["contador"] != MISSING]
print(f"\nCorrelación contador vs. reales (con campo): "
      f"{withField['contador'].astype(int).corr(withField['reales'].astype(int)):.2f}")

categoría
contador mayor            6699
sin campo, con reseñas    4154
contador menor            4107
coincide                  2449
sin campo, sin reseñas    2433

Correlación contador vs. reales (con campo): 0.36


In [54]:
print("Valores más frecuentes del contador:")
print(withField["contador"].value_counts().head(8).to_string())
print("\nLos 8 contadores más altos:")
display(withField.sort_values("contador", ascending=False).head(8))
print("Muestra al azar de películas que no coinciden:")
sample(withField[withField["categoría"] != "coincide"], 8)

Valores más frecuentes del contador:
contador
1      7164
2      3927
3      1426
4       382
5        94
6        17
393       9
405       8

Los 8 contadores más altos:


,title,contador,reales,categoría
19268,The Mask,456,18,contador mayor
3607,Dumb & Dumber,450,13,contador mayor
17492,The Unborn,447,5,contador mayor
7322,About a Boy,441,31,contador mayor
289,8 Mile,441,17,contador mayor
8517,Cinderella,439,30,contador mayor
8868,Shrek 2,439,17,contador mayor
7520,The Taking of Pelham 1 2 3,437,20,contador mayor


Muestra al azar de películas que no coinciden:


,title,contador,reales,categoría
18056,The Nightingale,2,0,contador mayor
13300,Edge of Darkness,1,4,contador menor
12500,The Noble Family,3,0,contador mayor
18305,Tango Cabaret,3,0,contador mayor
10666,Death of a Superhero,3,1,contador mayor
19242,"Hush...Hush, Sweet Charlotte",2,0,contador mayor
11392,Fight Club,417,19,contador mayor
17841,Bomber,2,0,contador mayor


### 4.4 `writers` cortados por comas

Algunos elementos de `writers` aparecen partidos: `'Claude Miller (scenario'` + `' adaptation and dialogue)'`.
El texto original era un solo string separado por comas, y al convertirlo en lista se cortó también en las comas
que están **dentro** de un paréntesis. No se perdió ningún carácter: se puede reconstruir sin adivinar (sin LLM),
uniendo los pedazos hasta que el paréntesis quede cerrado.

In [55]:
def isUnbalanced(s: str) -> bool:
    return s.count("(") != s.count(")")


def rejoinWriters(writers: list[str]) -> list[str]:
    """Une los pedazos de un paréntesis cortado por comas. Pega con ',' (sin espacio):
    el pedazo ya trae su espacio, y así un número como '20,000' queda intacto."""
    result, pending = [], None
    for piece in writers:
        pending = piece if pending is None else pending + "," + piece
        if pending.count("(") <= pending.count(")"):
            result.append(pending)
            pending = None
    if pending is not None:  # nunca cerró: se deja visible para que el chequeo lo cuente
        result.append(pending)
    return result


withWriters = [m for m in movies if isinstance(m.get("writers"), list)]
broken = [m for m in withWriters if any(isUnbalanced(w) for w in m["writers"])]
rejoined = {m["_id"]: rejoinWriters(m["writers"]) for m in broken}
closingNoSpace = [w for m in broken for w in m["writers"]
                  if w.count(")") > w.count("(") and not w.startswith(" ")]

display(pd.DataFrame([
    ("películas con writers (lista)", len(withWriters)),
    ("películas sin el campo", sum("writers" not in m for m in movies)),
    ("películas con algún elemento roto", len(broken)),
    ("elementos con paréntesis sin cerrar o sin abrir", sum(isUnbalanced(w) for m in broken for w in m["writers"])),
    ("películas que siguen rotas después de unir", sum(any(isUnbalanced(w) for w in r) for r in rejoined.values())),
    ("pedazos de cierre que no empiezan con espacio", len(closingNoSpace)),
], columns=["chequeo", "valor"]).set_index("chequeo"))
print("Pedazos de cierre sin espacio:", closingNoSpace)

for title in ["Thèrèse", "Twilight Zone: The Movie", "The Galapagos Affair: Satan Came to Eden"]:
    m = next(x for x in broken if x["title"].strip() == title)
    print(f"\n— {title}\n  antes:   {m['writers']}\n  después: {rejoined[m['_id']]}")

,valor
chequeo,
películas con writers (lista),18847
películas sin el campo,995
películas con algún elemento roto,46
elementos con paréntesis sin cerrar o sin abrir,168
películas que siguen rotas después de unir,0
pedazos de cierre que no empiezan con espacio,1


Pedazos de cierre sin espacio: ['000 Feet")']

— Thèrèse
  antes:   ['Franèois Mauriac (novel)', 'Claude Miller (scenario', ' adaptation and dialogue)', 'Natalie Carter (scenario', ' adaptation and dialogue)']
  después: ['Franèois Mauriac (novel)', 'Claude Miller (scenario, adaptation and dialogue)', 'Natalie Carter (scenario, adaptation and dialogue)']

— Twilight Zone: The Movie
  antes:   ['Rod Serling (television series The Twilight Zone)', 'John Landis (prologue)', 'John Landis (segment)', 'George Clayton Johnson (screenplay)', 'Richard Matheson (segment)', 'Melissa Mathison (segment)', 'Jerome Bixby (story)', 'Richard Matheson (segment)', 'Richard Matheson (short story "Nightmare at 20', '000 Feet")', 'Richard Matheson (screenplay)']
  después: ['Rod Serling (television series The Twilight Zone)', 'John Landis (prologue)', 'John Landis (segment)', 'George Clayton Johnson (screenplay)', 'Richard Matheson (segment)', 'Melissa Mathison (segment)', 'Jerome Bixby (story)', 'Richard M

### 4.5 Elementos repetidos en `writers`, `cast` y `directors`

Hay listas con el mismo elemento más de una vez: `['Steven Klein', 'Steven Klein', ...]`. En películas de episodios
(*Batman: Gotham Knight*) la repetición seguramente indicaba un crédito por episodio, pero esa aclaración no está en el
dataset, así que hoy no aporta nada.

**Regla:** se quita solo lo **idéntico** (texto exacto, rol incluido), se conserva la primera aparición y el orden.
`'Amy Albany (memoir)'` y `'Amy Albany'` son textos distintos y se quedan los dos: el rol es información.
En `writers` se aplica **después** de unir los pedazos (4.4), porque esa unión cambia qué elementos quedan repetidos.

**Limitación:** mismo texto no garantiza misma persona (homónimos, o nombres que la `è` volvió iguales). Como el texto
es idéntico, en pantalla tampoco se podrían distinguir: no se pierde nada recuperable.

In [56]:
def dropExactDuplicates(items: list) -> list:
    """Deja la primera aparición de cada elemento idéntico y respeta el orden."""
    return list(dict.fromkeys(items))


def personLists(field: str) -> list[tuple[dict, list]]:
    """(película, lista) para cada película con el campo como lista. writers va ya unido (regla de la 4.4)."""
    rows = []
    for m in movies:
        values = m.get(field)
        if isinstance(values, list):
            rows.append((m, rejoinWriters(values) if field == "writers" else values))
    return rows


rows = []
for field in ["writers", "cast", "directors"]:
    lists = personLists(field)
    withDups = [(m, v) for m, v in lists if len(v) != len(set(v))]
    deduped = [dropExactDuplicates(v) for _, v in withDups]
    rows.append({
        "campo": field,
        "listas": len(lists),
        "listas con repetidos": len(withDups),
        "elementos que se quitan": sum(len(v) - len(d) for (_, v), d in zip(withDups, deduped)),
        "siguen con repetidos": sum(len(d) != len(set(d)) for d in deduped),
        "quedan vacías": sum(len(d) == 0 for d in deduped),
    })
display(pd.DataFrame(rows).set_index("campo"))

examples = [
    ("writers", "Batman: Gotham Knight"),
    ("writers", "Print the Legend"),
    ("writers", "Low Down"),
    ("cast", "My Sweet Little Village"),
]
for field, title in examples:
    m, v = next((m, v) for m, v in personLists(field) if m["title"].strip() == title)
    print(f"\n— {title} ({field})\n  antes:   {v}\n  después: {dropExactDuplicates(v)}")

,listas,listas con repetidos,elementos que se quitan,siguen con repetidos,quedan vacías
campo,,,,,
writers,18847,125,167,0,0
cast,19638,2,2,0,0
directors,19604,3,3,0,0



— Batman: Gotham Knight (writers)
  antes:   ['Bob Kane (creator)', 'Jordan Goldberg (story)', 'Josh Olson (screenplay)', 'Jordan Goldberg (story)', 'Greg Rucka (screenplay)', 'Jordan Goldberg (story)', 'Jordan Goldberg (screenplay)', 'Jordan Goldberg (story)', 'David S. Goyer (screenplay)', 'Jordan Goldberg (story)', 'Brian Azzarello (screenplay)', 'Jordan Goldberg (story)', 'Alan Burnett (screenplay)']
  después: ['Bob Kane (creator)', 'Jordan Goldberg (story)', 'Josh Olson (screenplay)', 'Greg Rucka (screenplay)', 'Jordan Goldberg (screenplay)', 'David S. Goyer (screenplay)', 'Brian Azzarello (screenplay)', 'Alan Burnett (screenplay)']

— Print the Legend (writers)
  antes:   ['Steven Klein', 'Steven Klein', 'Luis Lopez', 'Luis Lopez', 'J. Clay Tweel', 'J. Clay Tweel']
  después: ['Steven Klein', 'Luis Lopez', 'J. Clay Tweel']

— Low Down (writers)
  antes:   ['Amy Albany (memoir)', 'Topper Lilien (screenplay)', 'Amy Albany (screenplay)', 'Topper Lilien', 'Amy Albany (memoir)', 'Am

### 4.6 `year` vs. `released`

`released` no es el primer estreno: es **una** fecha de estreno, que puede ser en otro país o un reestreno.
*Godzilla* (1954, Ishirō Honda) tiene `released` 2004; los remakes de 1998 y 2014 son otros documentos, con otro
`imdb.id` y otro director. Que los dos campos difieran no es un error: miden cosas distintas.

**Decisión:** `year` es el año de la película (el que se muestra). `released` se conserva tal cual y en el contrato
se documenta como "una fecha de estreno, no necesariamente la primera". Los 3 casos con `released` **anterior** a
`year` también se dejan: dos son plausibles (festival antes del estreno comercial) y *The World According to Sesame
Street* (1996 vs. 2006) parece un error, pero el dataset solo no alcanza para demostrarlo (mismo criterio que la `è`).

In [57]:
comparable = [m for m in movies
              if isinstance(m.get("year"), int) and m["year"] > 0 and hasattr(m.get("released"), "year")]
yearDiff = pd.DataFrame([{"dif": m["released"].year - m["year"], "year": m["year"], "released": m["released"].date(),
                          "title": m["title"], "countries": m.get("countries"), "imdbId": m.get("imdb", {}).get("id")}
                         for m in comparable])

print(f"películas: {len(movies)} | comparables (year entero y released Date): {len(comparable)}")
print(f"released.year - year > 1: {(yearDiff['dif'] > 1).sum()} | negativas: {(yearDiff['dif'] < 0).sum()}")
bins = pd.cut(yearDiff["dif"], [-100, -1, 0, 1, 2, 5, 10, 20, 50, 100])
display(bins.value_counts(sort=False).rename("películas").to_frame())

# Los extremos (>= 50 años) y todos los negativos, sin recortar
display(yearDiff[(yearDiff["dif"] >= 50) | (yearDiff["dif"] < 0)].sort_values("dif", ascending=False))

# Hipótesis "son remakes": los remakes son otros documentos (otro imdb.id y otro director)
display(pd.DataFrame([{"title": m["title"], "year": m.get("year"), "released": m.get("released"),
                       "directors": m.get("directors"), "imdbId": m.get("imdb", {}).get("id")}
                      for m in movies if m["title"] == "Godzilla"]))

películas: 19842 | comparables (year entero y released Date): 19050
released.year - year > 1: 1060 | negativas: 3


,películas
dif,
"(-100, -1]",3
"(-1, 0]",14102
"(0, 1]",3885
"(1, 2]",652
"(2, 5]",264
"(5, 10]",67
"(10, 20]",39
"(20, 50]",34
"(50, 100]",4


,dif,year,released,title,countries,imdbId
17224,78,1932,2010-05-20,The Blood of a Poet,[France],21331
5481,61,1944,2005-04-06,The Battle of China,[USA],184254
13761,60,1934,1994-01-01,A Story of Floating Weeds,[Japan],25929
16874,58,1943,2001-10-27,Meshes of the Afternoon,[USA],36154
8046,50,1954,2004-05-07,Godzilla,[Japan],47034
14632,50,1949,1999-07-01,Begone Dull Care,[Canada],41169
12919,-1,2004,2003-09-20,Purgatory House,[USA],298120
15214,-4,2010,2006-02-06,Who Is Harry Nilsson (And Why Is Everybody Talkin' About Him?),[USA],756727
17312,-10,2006,1996-01-19,The World According to Sesame Street,[USA],434440


,title,year,released,directors,imdbId
0,Godzilla,1998,1998-05-20,[Roland Emmerich],120685
1,Godzilla,2014,2014-05-16,[Gareth Edwards],831387
2,Godzilla,1954,2004-05-07,[Ishirè Honda],47034


### 4.7 Vocabulario de `rated`

`rated` es la calificación por edades. Tiene dos problemas de escritura y una mezcla de sistemas:

- **Mayúsculas:** `'Approved'` (2) y `'Not Rated'` (1) son `'APPROVED'` y `'NOT RATED'` escritos distinto.
- **Sinónimos:** `NOT RATED` y `UNRATED` dicen lo mismo ("no tiene calificación").
- **Sistemas de varios países:** `X` aparece hasta 1990 en películas que en EE. UU. salieron como `R`
  (*Friday the 13th*): seguramente es la `X` británica ("mayores de 18"), no la estadounidense. `M` aparece hasta
  2012, aunque en EE. UU. dejó de existir en 1969. El dataset no dice de qué país es cada valor.

**Decisión:** pasar a mayúsculas y unificar `UNRATED` → `NOT RATED`. Los códigos viejos o de otros países (`X`, `M`,
`GP`, `APPROVED`, `PASSED`…) **se dejan como vienen**: convertirlos sin saber el país inventaría datos
(*Friday the 13th* quedaría como `NC-17`). `NOT RATED` ("no tiene calificación") no se mezcla con la falta del campo
("no sabemos").

In [58]:
def normalizeRated(value: str) -> str:
    upper = value.upper()
    return "NOT RATED" if upper == "UNRATED" else upper


ratedRaw = Counter(m["rated"] if "rated" in m else "<sin campo>" for m in movies)
ratedClean = Counter(normalizeRated(m["rated"]) if "rated" in m else "<sin campo>" for m in movies)
display(pd.DataFrame({"antes": pd.Series(ratedRaw), "después": pd.Series(ratedClean)})
        .fillna(0).astype(int).sort_values("antes", ascending=False))

# Variantes de mayúsculas, todas
display(pd.DataFrame([{"rated": m["rated"], "year": m.get("year"), "title": m["title"]}
                      for m in movies if m.get("rated") in ("Approved", "Not Rated")]))

# Años por código: X y M aparecen después de que EE. UU. los dejara de usar
yearsByRated = pd.DataFrame([{"rated": m["rated"], "year": m["year"]} for m in movies
                             if "rated" in m and isinstance(m.get("year"), int) and m["year"] > 0])
codes = ["APPROVED", "PASSED", "M", "GP", "X", "NC-17"]
display(yearsByRated[yearsByRated["rated"].isin(codes)].groupby("rated")["year"]
        .agg(["count", "min", "median", "max"]).loc[codes])
print("Con X:", sorted({m["title"] for m in movies if m.get("rated") == "X"}))

,antes,después
<sin campo>,6515,6515
R,5453,5453
PG-13,2299,2299
PG,1827,1827
NOT RATED,1315,2022
UNRATED,706,0
APPROVED,683,685
G,467,467
PASSED,163,163
TV-14,83,83


,rated,year,title
0,Approved,1939,Tevya
1,Not Rated,2012,Pilgrim Song
2,Approved,1949,The Hidden Room


,count,min,median,max
rated,,,,
APPROVED,664,1924,1954.0,1992
PASSED,160,1915,1938.0,1964
M,32,1953,1969.0,2012
GP,42,1966,1970.5,1972
X,33,1967,1976.0,1990
NC-17,38,1972,2001.5,2013


Con X: ['A Clockwork Orange', 'A Nightmare on Elm Street 3: Dream Warriors', 'Angel Heart', 'Arabian Nights', 'Assault on Precinct 13', 'Bad Timing', 'Deep Red', "Dionysus in '69", 'Dressed to Kill', 'Drive, He Said', 'End of the Road', 'Evil Dead II', 'Flesh Gordon', 'Flesh+Blood', 'Friday the 13th', 'Greetings', 'I Am Curious (Yellow)', 'I Spit on Your Grave', 'If....', 'In the Realm of the Senses', 'King of New York', 'Leatherface: Texas Chainsaw Massacre III', 'Midnight Cowboy', 'Ms .45', 'Phenomena', 'Salon Kitty', 'Suspiria', 'The Canterbury Tales', 'The Devils', 'The Hills Have Eyes', 'The Texas Chainsaw Massacre 2']


### 4.8 `lastupdated` como texto en 2 formatos (decisión A7)

`lastupdated` viene como **texto**, con fracción de segundo (`'2015-08-31 21:51:13.147000000'`) o sin ella
(`'2015-08-22 00:16:37'`). Se usa para elegir la copia más reciente al deduplicar (A5), y la decisión A7 fue
convertirlo a fecha antes de comparar, porque comparar texto es comparar letra por letra.

**Verificación:** en los 651 grupos de duplicados, comparar como texto o como fecha elige **la misma copia**: el riesgo
no ocurre en este dataset, pero convertir sigue siendo lo seguro. La fracción tiene 9 dígitos, pero los últimos 6 son
siempre `0`: la precisión real es de milisegundos, la misma que guarda un `Date` de Mongo (no se pierde nada).

**Decisión:** en la base limpia, `lastupdated` se guarda como `Date`.

In [59]:
lastUpdatedPattern = Counter("con fracción" if "." in m["lastupdated"] else "sin fracción" for m in movies)
lastUpdatedDate = pd.to_datetime(pd.Series([m["lastupdated"] for m in movies]), format="ISO8601")
fractions = [m["lastupdated"].split(".")[1] for m in movies if "." in m["lastupdated"]]

copiesByImdb = defaultdict(list)
for m in movies:
    copiesByImdb[m.get("imdb", {}).get("id")].append(m["lastupdated"])
dupGroups = [v for v in copiesByImdb.values() if len(v) > 1]
mixedGroups = [v for v in dupGroups if any("." in s for s in v) and any("." not in s for s in v)]
disagree = [v for v in dupGroups if pd.Timestamp(max(v)) != max(pd.Timestamp(s) for s in v)]

display(pd.DataFrame([
    ("con fracción", lastUpdatedPattern["con fracción"]),
    ("sin fracción", lastUpdatedPattern["sin fracción"]),
    ("se pueden convertir a fecha", int(lastUpdatedDate.notna().sum())),
    ("fracciones cuyos últimos 6 dígitos son 0", sum(f.endswith("000000") for f in fractions)),
    ("grupos de duplicados", len(dupGroups)),
    ("grupos con los 2 formatos mezclados", len(mixedGroups)),
    ("grupos donde texto y fecha eligen distinta copia", len(disagree)),
], columns=["chequeo", "valor"]).set_index("chequeo"))
print(f"Rango: {lastUpdatedDate.min()} → {lastUpdatedDate.max()}")
print("Grupos con formatos mezclados:", mixedGroups)

,valor
chequeo,
con fracción,19764
sin fracción,78
se pueden convertir a fecha,19842
fracciones cuyos últimos 6 dígitos son 0,19764
grupos de duplicados,651
grupos con los 2 formatos mezclados,1
grupos donde texto y fecha eligen distinta copia,0


Rango: 2014-02-27 00:47:05.047000 → 2015-09-17 17:31:50.663000
Grupos con formatos mezclados: [['2015-08-22 00:16:37', '2015-08-31 21:51:13.147000000']]


### 4.9 Pósters (`poster`)

Las 19.842 películas tienen `poster`: siempre una URL a un `.jpg`, ninguna vacía ni con otro tipo. Todas apuntan a
`https://m.media-amazon.com` salvo **una**: *Sikumi (On the Ice)* (2008, `imdb.id` 1152712), que usa el servidor viejo
de imágenes de IMDb, `http://ia.media-imdb.com`.

**Prueba por red (hecha a mano el 2026-11-19, no se repite en este notebook para que sea reproducible sin internet):**

| URL | Respuesta |
| :-- | :-- |
| `http://ia.media-imdb.com/images/M/MV5BMTI5…@@._V1_SX300.jpg` (original) | `403`, página HTML de error: en el frontend se vería rota |
| `https://m.media-amazon.com/images/M/MV5BMTI5…@@._V1_SX300.jpg` (misma ruta, servidor nuevo) | `200`, `image/jpeg` de 29 KB |

La imagen recuperada **no es un póster propio** de *Sikumi*: es la tapa del DVD *Cinema16: World Short Films*, una
recopilación de cortos que incluye el de Andrew Okpeaha MacLean (director de *Sikumi*). Es la misma imagen que la
fuente le asignó (misma ruta), así que se conserva, igual que `released` en A19.

**URLs rotas (muestra hecha a mano el 2026-11-19, tampoco se repite acá):** un formato correcto **no garantiza** que la
imagen exista. Sobre 300 películas únicas elegidas al azar (semilla 2026, de 19.145), **35 dan `404`** (≈12%; con ese
tamaño de muestra, entre 8% y 15%: unas 1.500 a 2.900 películas). No son películas raras (*GoldenEye*, *A Passage to
India*). Probando otros tamaños de la misma imagen (`._V1_SX300.jpg`, `._V1_.jpg`, `.jpg`) también da `404`: Amazon borró
la imagen, así que no hay regla que la recupere.

**Decisión A24:** el `cleaning-agent` **no** verifica ni anula pósters: una URL que hoy funciona puede romperse mañana, así
que el problema se resuelve donde se usa: el frontend muestra un póster genérico si la imagen no carga. Descartadas:
probar las 19.842 en la limpieza y poner `null` (se desactualiza y el frontend necesita el respaldo igual) y buscar los
pósters en una fuente externa por `imdb.id` (ej. TMDB: queda como **pregunta para la cátedra / mejora futura**).

**Imágenes compartidas:** 12 URLs se repiten en películas con `imdb.id` distinto. No es un error: son sagas (*Red Riding*,
*The Human Condition*, *Seven Up!*), tapas de DVD que traen ambas películas (*Aftermath / Genesis*, *Primary* y *Crisis*,
*Gaea Girls / Shinjuku Boys*) o pares que ya son duplicados (2.7: *Confessions of a Dangerous Mind*, *Divide and Conquer*,
*The Kingdom*, *Eros*). Dudoso: *Sleeping Betty* / *The Necktie* (tapa de *Animation Express*; solo figura la primera).
*The Pedestrian* / *Tales from the Vienna Woods* comparten una URL que da `404`. No se corrige; se documenta.

**Decisión A23:** si el servidor es `ia.media-imdb.com`, se reemplaza por `https://m.media-amazon.com` dejando el resto de
la URL igual. Se documenta que el póster de *Sikumi* es la tapa de una recopilación.

In [60]:
posterStatus = Counter()
posterHosts = Counter()
for m in movies:
    poster = m.get("poster")
    if poster is None:
        posterStatus["sin campo"] += 1
    elif not isinstance(poster, str):
        posterStatus[f"tipo {type(poster).__name__}"] += 1
    elif not poster.strip():
        posterStatus["string vacío"] += 1
    else:
        posterStatus["URL a .jpg" if poster.lower().endswith(".jpg") else "URL a otro formato"] += 1
        posterHosts[poster.split("/images/")[0]] += 1

display(pd.DataFrame(posterStatus.items(), columns=["estado", "películas"]).set_index("estado"))
display(pd.DataFrame(posterHosts.most_common(), columns=["servidor", "películas"]).set_index("servidor"))

oldHost = "http://ia.media-imdb.com"
display(pd.DataFrame(
    [(m["title"], m.get("year"), m["imdb"]["id"], m["poster"]) for m in movies if m["poster"].startswith(oldHost)],
    columns=["title", "year", "imdb.id", "poster"],
))

# Verificación de A23: después de reemplazar el servidor, ¿queda alguna URL fuera de m.media-amazon.com?
newPosters = [m["poster"].replace(oldHost, "https://m.media-amazon.com", 1) for m in movies]
print("Fuera de https://m.media-amazon.com después de A23:",
      sum(not p.startswith("https://m.media-amazon.com/") for p in newPosters))

,películas
estado,
URL a .jpg,19842


,películas
servidor,
https://m.media-amazon.com,19841
http://ia.media-imdb.com,1


,title,year,imdb.id,poster
0,Sikumi (On the Ice),2008,1152712,http://ia.media-imdb.com/images/M/MV5BMTI5OTY2OTMwMV5BMl5BanBnXkFtZTcwODIzODAxMw@@._V1_SX300.jpg


Fuera de https://m.media-amazon.com después de A23: 0


### 4.10 `awards`: los números no coinciden con el texto (decisión A25)

Cada película tiene `awards.text` (una frase) y dos números, `wins` y `nominations`. Cuando la frase empieza con un premio
grande (`'Won 1 Oscar.'` o `'Nominated for 2 Oscars.'`) seguido de `'Another …'`, ese número se sumó **al primer número que
aparece después de *Another***, sin mirar si era una victoria o una nominación:

- *Blade Runner*: `'Nominated for 2 Oscars. Another 11 wins & 15 nominations.'` → guardado `wins: 13` (las 2 nominaciones se
  contaron como premios ganados). Correcto: `wins: 11, nominations: 17`.
- *The Wrong Box*: `'Won 1 BAFTA Film Award. Another 2 nominations.'` → guardado `wins: 0, nominations: 3`. Correcto: `1` y `2`.

En los otros dos casos (*ganó* + "Another N wins", *nominada* + "Another N nominations") la regla coincide por casualidad
con lo correcto. El mecanismo se verifica abajo sobre **todas** las películas, no solo en los ejemplos.

**Decisión A25:** el `cleaning-agent` **recalcula `wins` y `nominations` a partir de `awards.text`**: `Won N` suma a `wins`,
`Nominated for N` suma a `nominations`, y se suman los números de la parte "Another …". El texto se deja como está. No se
adivina nada (mismo criterio que A17): el dato correcto está en el mismo documento. Si una frase no se pudiera leer, los
números quedan como están y se registra en la auditoría.

In [61]:
# La frase completa tiene hasta 3 partes, todas opcionales; re.VERBOSE permite comentar el patrón por partes.
AWARDS_PATTERN = re.compile(r"""
    ^(?:
        (?P<verb>Won|Nominated\ for)\ (?P<big>\d+)\ [^.]+\.   # 1) premio grande: "Won 1 Oscar." / "Nominated for 2 Oscars."
        (?:\ Another\ )?                                     #    seguido (o no) de " Another "
    )?
    (?:(?P<wins>\d+)\ wins?)?                                  # 2) "11 wins" / "1 win"
    (?:\ &\ )?
    (?:(?P<noms>\d+)\ nominations?)?                           # 3) "15 nominations" / "1 nomination"
    \.?$
""", re.VERBOSE)

def parseAwards(text: str):
    """Devuelve (verb, big, wins, noms) leídos de la frase, o None si no se entiende."""
    match = AWARDS_PATTERN.match(text.strip())
    if not match or not any(match.groups()):
        return None
    g = match.groupdict()
    return g["verb"], int(g["big"] or 0), int(g["wins"] or 0), int(g["noms"] or 0)

def correctAwards(verb, big, wins, noms):
    return wins + (big if verb == "Won" else 0), noms + (big if verb == "Nominated for" else 0)

def buggyAwards(verb, big, wins, noms):
    # Mecanismo sospechado: el premio grande se suma al primer número después de "Another"
    if not verb:
        return wins, noms
    return (wins + big, noms) if wins else (wins, noms + big)

awardsRows = []
for m in movies:
    parsed = parseAwards(m["awards"]["text"])
    stored = (m["awards"]["wins"], m["awards"]["nominations"])
    if parsed is None:
        awardsRows.append(("no se pudo leer", m["title"], m["awards"]["text"], stored, None))
        continue
    verb = parsed[0] or "sin premio grande"
    status = "coincide" if correctAwards(*parsed) == stored else "mal calculado"
    awardsRows.append((f"{verb} → {status}", m["title"], m["awards"]["text"], stored, correctAwards(*parsed),
                       buggyAwards(*parsed) == stored))

awardsDf = pd.DataFrame(awardsRows, columns=["patrón", "title", "text", "guardado", "recalculado", "explicadoPorMecanismo"])
display(awardsDf.groupby("patrón").agg(películas=("title", "size"), ejemplo=("title", "first")))
display(pd.DataFrame([
    ("frases que no se pudieron leer", int((awardsDf["recalculado"].isna()).sum())),
    ("valor guardado explicado por el mecanismo", int(awardsDf["explicadoPorMecanismo"].sum())),
    ("películas que cambian al recalcular", int((awardsDf["guardado"] != awardsDf["recalculado"]).sum())),
], columns=["chequeo", "valor"]).set_index("chequeo"))
awardsDf[awardsDf["title"].isin(["Blade Runner", "The Wrong Box", "Interstellar", "Carrie"])].drop_duplicates("title")[
    ["title", "text", "guardado", "recalculado"]]

,películas,ejemplo
patrón,,
Nominated for → coincide,412,Carrie
Nominated for → mal calculado,2407,Blade Runner
Won → coincide,1084,Interstellar
Won → mal calculado,170,The Wrong Box
sin premio grande → coincide,15769,A Fish Tale


,valor
chequeo,
frases que no se pudieron leer,0
valor guardado explicado por el mecanismo,19842
películas que cambian al recalcular,2577


,title,text,guardado,recalculado
2,Blade Runner,Nominated for 2 Oscars. Another 11 wins & 15 nominations.,"(13, 15)","(11, 17)"
8,Interstellar,Won 1 Oscar. Another 44 wins & 100 nominations.,"(45, 100)","(45, 100)"
13,Carrie,Nominated for 2 Oscars. Another 3 nominations.,"(0, 5)","(0, 5)"
170,The Wrong Box,Won 1 BAFTA Film Award. Another 2 nominations.,"(0, 3)","(1, 2)"


### 4.11 `tomatoes`: bloque `critic` faltante o incompleto (decisión A26)

Rotten Tomatoes clasifica cada crítica como **fresh** (positiva) o **rotten** (negativa). `critic.numReviews` es la cantidad
de críticas, `critic.meter` el porcentaje de positivas (Tomatometer) y `critic.rating` el promedio de 0 a 10.

- **258 películas** tienen `fresh`/`rotten` pero **no** el bloque `critic` (con 1 a 112 críticas).
- **38** tienen `critic` con solo `numReviews` (1 a 4 críticas): probablemente Rotten Tomatoes no publica el porcentaje con
  tan pocas, pero el dataset no permite comprobarlo.
- `meter` = `fresh / (fresh + rotten) × 100` en **11.399 de 11.400**. La excepción: *Around the Block* (2013), `meter: 57`
  con 5 fresh y 3 rotten (daría 62,5) y `rating: 0`.

**Decisión A26:** no se modifica nada, se documenta. En las 258 + 38 el dato **falta**, no está mal: calcular `meter` sería
completar algo que la fuente no publicó, y `rating` no se puede calcular. *Around the Block* también se deja como está
(decisión del usuario, aunque contradice la fórmula). **Pendiente para el grupo:** qué campos de `movies` muestra la ficha y
cuáles usa el recomendador; si `tomatoes` se usa, se puede revisar.

In [62]:
tomatoesMovies = [m["tomatoes"] for m in movies if "tomatoes" in m]
tomatoesShape = Counter(("con critic" if "critic" in t else "sin critic", "con fresh/rotten" if "fresh" in t else "sin fresh/rotten")
                        for t in tomatoesMovies)
display(pd.DataFrame([(a, b, n) for (a, b), n in tomatoesShape.items()], columns=["critic", "fresh/rotten", "películas"]))

withoutCritic = [t for t in tomatoesMovies if "fresh" in t and "critic" not in t]
print("Sin critic: fresh + rotten entre",
      min(t["fresh"] + t["rotten"] for t in withoutCritic), "y", max(t["fresh"] + t["rotten"] for t in withoutCritic))

withCritic = [(m["title"], m.get("year"), m["tomatoes"]) for m in movies if "critic" in m.get("tomatoes", {})]
incomplete = [(title, t["critic"]["numReviews"]) for title, _, t in withCritic if "meter" not in t["critic"]]
print("critic con solo numReviews:", len(incomplete), "| numReviews entre",
      min(n for _, n in incomplete), "y", max(n for _, n in incomplete))

# ¿meter es el porcentaje de críticas positivas? (tolerancia de 1 punto por redondeo)
withMeter = [(title, year, t) for title, year, t in withCritic if "meter" in t["critic"]]
meterMismatch = [(title, year, t["critic"], t["fresh"], t["rotten"]) for title, year, t in withMeter
                 if abs(t["critic"]["meter"] - 100 * t["fresh"] / (t["fresh"] + t["rotten"])) > 1]
print(f"meter ≈ fresh/(fresh+rotten): {len(withMeter) - len(meterMismatch)} de {len(withMeter)}")
pd.DataFrame(meterMismatch, columns=["title", "year", "critic", "fresh", "rotten"])

,critic,fresh/rotten,películas
0,sin critic,sin fresh/rotten,5863
1,con critic,con fresh/rotten,11438
2,sin critic,con fresh/rotten,258


Sin critic: fresh + rotten entre 1 y 112
critic con solo numReviews: 38 | numReviews entre 1 y 4
meter ≈ fresh/(fresh+rotten): 11399 de 11400


,title,year,critic,fresh,rotten
0,Around the Block,2013,"{'rating': 0, 'numReviews': 8, 'meter': 57}",5,3


### 4.12 `tomatoes.dvd`: fechas imposibles (decisión A27)

El DVD salió a la venta en 1996-1997 y una película no puede salir en DVD antes de existir. Sin embargo, `tomatoes.dvd`
tiene 330 fechas anteriores a 1996 (incluida *Odd Girl Out*, de 2005, con `1910-01-10`) y 89 películas cuyo DVD
es anterior a su propio `year`. 15 de estas tienen exactamente `1998-01-01`, que parece un valor de relleno. Las de
1991-1995 podrían ser otro formato (VHS) cargado en el mismo campo, pero el dataset no lo dice.

**Decisión A27:** igual que el resto de `tomatoes` (A26): **no se modifica**; se documenta que `tomatoes.dvd` **no es
confiable**. Se suma a la pregunta al grupo sobre qué campos se usan (el usuario cree que no se va a usar).

In [63]:
dvdMovies = [m for m in movies if "dvd" in m.get("tomatoes", {})]
dvdBefore1998 = Counter(m["tomatoes"]["dvd"].year for m in dvdMovies if m["tomatoes"]["dvd"].year < 1998)
display(pd.DataFrame(sorted(dvdBefore1998.items()), columns=["año del DVD", "películas"]).set_index("año del DVD").T)

dvdBeforeFilm = [(m["title"], m["year"], m["tomatoes"]["dvd"].date()) for m in dvdMovies
                 if isinstance(m.get("year"), int) and m["tomatoes"]["dvd"].year < m["year"]]
print(f"Con dvd: {len(dvdMovies)} | anteriores a 1996: {sum(n for y, n in dvdBefore1998.items() if y < 1996)}"
      f" | DVD anterior al year de la película: {len(dvdBeforeFilm)}")
print("De esas, con exactamente 1998-01-01:",
      sum(str(d) == "1998-01-01" for _, _, d in dvdBeforeFilm))
pd.DataFrame(dvdBeforeFilm, columns=["title", "year", "dvd"]).sort_values("dvd").head(10)

año del DVD,1910,1950,1978,1979,1984,1986,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997
películas,1,1,1,1,1,3,11,4,17,65,71,54,55,45,66,208


Con dvd: 14262 | anteriores a 1996: 330 | DVD anterior al year de la película: 89
De esas, con exactamente 1998-01-01: 15


,title,year,dvd
36,Odd Girl Out,2005,1910-01-10
8,"Refugee All Stars, Sierra Leone's",2005,1988-01-01
30,Gia,1998,1991-12-23
15,Dottie Gets Spanked,1993,1992-01-01
50,The Love Letter,1998,1997-11-28
58,Book of Shadows: Blair Witch 2,2000,1998-01-01
32,Downfall,2004,1998-01-01
37,Downfall,2004,1998-01-01
39,Together,2000,1998-01-01
40,A Decade Under the Influence,2003,1998-01-01


### 4.13 Campos revisados sin problemas

Registro de lo que se revisó en `movies` y **no** mostró anomalías, para que el `cleaning-agent` no tenga reglas sobre
estos campos y quede constancia de que se miraron. Los campos de `reviews` se revisaron en la sección 3.

In [64]:
def nested(doc, path):
    for key in path.split("."):
        if not isinstance(doc, dict) or key not in doc:
            return None
        doc = doc[key]
    return doc

def valueRange(path):
    values = [v for v in (nested(m, path) for m in movies) if v is not None]
    return f"{len(values)} valores, de {min(values)} a {max(values)}"

listFields = ["cast", "countries", "directors", "genres", "languages", "writers"]
textFields = ["title", "plot", "fullplot"]
withCriticBlock = [m["tomatoes"] for m in movies if "critic" in m.get("tomatoes", {})]

checks = [
    *[(f"{f}: vacíos o solo espacios", sum(not m[f].strip() for m in movies if f in m)) for f in textFields],
    *[(f"{f}: listas vacías", sum(isinstance(m.get(f), list) and not m[f] for m in movies)) for f in listFields],
    *[(f"{f}: elementos vacíos", sum(isinstance(m.get(f), list) and any(not str(x).strip() for x in m[f])
                                     for m in movies)) for f in listFields],
    ("type: valores", dict(Counter(m["type"] for m in movies))),
    ("metacritic (0–100)", valueRange("metacritic")),
    ("awards.wins (≥ 0)", valueRange("awards.wins")),
    ("awards.nominations (≥ 0)", valueRange("awards.nominations")),
    ("tomatoes.critic.meter (0–100)", valueRange("tomatoes.critic.meter")),
    ("tomatoes.critic.rating (0–10)", valueRange("tomatoes.critic.rating")),
    ("tomatoes.viewer.meter (0–100)", valueRange("tomatoes.viewer.meter")),
    ("tomatoes.viewer.rating (0–5)", valueRange("tomatoes.viewer.rating")),
    ("fresh + rotten ≠ critic.numReviews", sum(t["fresh"] + t["rotten"] != t["critic"]["numReviews"] for t in withCriticBlock)),
]
pd.DataFrame(checks, columns=["chequeo", "resultado"]).set_index("chequeo")

,resultado
chequeo,
title: vacíos o solo espacios,0
plot: vacíos o solo espacios,0
fullplot: vacíos o solo espacios,0
cast: listas vacías,0
countries: listas vacías,0
directors: listas vacías,0
genres: listas vacías,0
languages: listas vacías,0
writers: listas vacías,0


## 5. Interacciones entre reglas (Paso 5)

Qué pasa cuando las reglas de las secciones 2 a 4 se aplican juntas. El orden ya está decidido (A8): limpiar campos →
deduplicar películas → reasignar reseñas. Igual que el resto del notebook, esto **solo mide**: no modifica el dataset.

### 5.1 Reseñas de películas duplicadas

Al deduplicar queda una copia ganadora por película (A5/A6: `lastupdated` más reciente; desempate por más campos
completos y después `_id` mayor) y las perdedoras van a soft delete (G2). Las reseñas que apuntan a una copia perdedora
se **reasignan** a la ganadora. Grupos: los de A3 (`imdb.id`) más los 4 pares de A4 (2.7).

*Aproximación:* "más campos completos" se cuenta como campos de primer nivel no vacíos. Si la definición final cambia,
puede cambiar **qué** reseñas se mueven, pero no el total de reseñas sobre películas duplicadas.

In [65]:
# Grupos de duplicados: A3 (mismo imdb.id) + A4 (los pares que encontró la regla secundaria en 2.7)
movieGroups = defaultdict(list)
for m in movies:
    movieGroups[m["imdb"]["id"]].append(m)
for idA, idB in ruleA4.loc[ruleA4["regla A4"], "imdb.ids"]:
    movieGroups[idA] += movieGroups.pop(idB)
duplicateGroups = {k: g for k, g in movieGroups.items() if len(g) > 1}

def filledFields(m):
    return sum(v not in (None, "", [], {}) for v in m.values())

def winnerCopy(copies):
    return max(copies, key=lambda m: (pd.Timestamp(m["lastupdated"]), filledFields(m), m["_id"]))

survivorOf = {}
for copies in duplicateGroups.values():
    winner = winnerCopy(copies)
    for m in copies:
        survivorOf[m["_id"]] = winner["_id"]

onDuplicates = [r for r in reviews if r["movie_id"] in survivorOf]
toReassign = [r for r in onDuplicates if survivorOf[r["movie_id"]] != r["movie_id"]]
copiesWithReviews = defaultdict(set)
for r in onDuplicates:
    copiesWithReviews[survivorOf[r["movie_id"]]].add(r["movie_id"])

pd.DataFrame([
    ("grupos de duplicados (A3 + A4)", len(duplicateGroups)),
    ("copias perdedoras (soft delete)", sum(len(g) - 1 for g in duplicateGroups.values())),
    ("reseñas sobre películas duplicadas", len(onDuplicates)),
    ("· ya apuntan a la copia ganadora", len(onDuplicates) - len(toReassign)),
    ("· apuntan a una perdedora → reasignar", len(toReassign)),
    ("grupos con reseñas repartidas en 2+ copias", sum(len(s) > 1 for s in copiesWithReviews.values())),
], columns=["chequeo", "valor"]).set_index("chequeo")

,valor
chequeo,
grupos de duplicados (A3 + A4),655
copias perdedoras (soft delete),701
reseñas sobre películas duplicadas,4723
· ya apuntan a la copia ganadora,2249
· apuntan a una perdedora → reasignar,2474
grupos con reseñas repartidas en 2+ copias,329


### 5.2 Votos duplicados después de reasignar (decisión B13)

Al reasignar, un usuario que votó **dos copias** de la misma película queda con dos votos sobre ella. Además, la regla de B1
("gana el más reciente") se verificó solo con fechas comparables: algunos votos duplicados tienen fechas que B9/B10
anulan (`yesterday`, futuras) o que solo dicen el año (`circa`).

**Decisión B13** (reemplaza el criterio de B1 para **todos** los votos duplicados):
1. Solo cuentan las **fechas válidas**: posteriores al estreno y no posteriores a la creación del dataset. `yesterday` y
   las futuras no cuentan; `circa N` cuenta como 1/1/N.
2. Gana la **fecha válida más reciente**; el resto, soft delete.
3. Si empatan (misma fecha o ninguna válida), gana el que empieza con `"Update: Changed my mind…"`.
4. Si sigue el empate (copias idénticas), gana el `_id` mayor.

Consecuencia aceptada: si el "Update" tiene fecha futura, **pierde** (la fecha manda sobre el texto). Se descartó que el
"Update" siempre gane y promediar los votos (inventa un rating que el usuario no dio).
*Límite:* `circa 2023` frente a una fecha exacta de 2023 no se puede ordenar bien; se cuenta abajo.

In [66]:
def parseReviewDate(d):
    """Fecha de la reseña según B2 (dd/mm, ISO, epoch en ms) y B9 ('circa N' → 1/1/N). None si no se sabe."""
    if isinstance(d, datetime):
        return d.replace(tzinfo=None)
    if isinstance(d, int):
        return pd.to_datetime(d, unit="ms").to_pydatetime()
    if isinstance(d, str):
        if found := re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", d):
            return datetime(int(found[3]), int(found[2]), int(found[1]))
        if found := re.fullmatch(r"(\d{4})-(\d{2})-(\d{2})", d):
            return datetime(int(found[1]), int(found[2]), int(found[3]))
        if found := re.fullmatch(r"circa (\d{4})", d):
            return datetime(int(found[1]), 1, 1)
    return None

movieById = {m["_id"]: m for m in movies}

def validDate(r):
    d = parseReviewDate(r["date"])
    released = movieById.get(r["movie_id"], {}).get("released")
    if d is None or d > datasetCreated or (isinstance(released, datetime) and d < released):
        return None
    return d

def isUpdate(r):
    return r["comment"].startswith(UPDATE_PREFIX)

def winnerVote(votes):
    # max() sobre la tupla aplica los 4 pasos en orden: fecha válida, más reciente, "Update", _id mayor
    return max(votes, key=lambda r: (validDate(r) is not None, validDate(r) or datetime.min, isUpdate(r), r["_id"]))

identifiedReviews = [r for r in reviews if r["user_id"] not in ANONYMOUS_IDS]
votesBefore, votesAfter = defaultdict(list), defaultdict(list)
for r in identifiedReviews:
    votesBefore[(r["user_id"], r["movie_id"])].append(r)
    votesAfter[(r["user_id"], survivorOf.get(r["movie_id"], r["movie_id"]))].append(r)
dupBefore = {k: v for k, v in votesBefore.items() if len(v) > 1}
dupAfter = {k: v for k, v in votesAfter.items() if len(v) > 1}
fromCopies = {k: v for k, v in dupAfter.items() if len({r["movie_id"] for r in v}) > 1}

def dateKind(r):
    d = parseReviewDate(r["date"])
    if d is None:
        return "sin fecha"
    if d > datasetCreated:
        return "futura"
    return "circa" if str(r["date"]).startswith("circa") else "válida"

kinds = Counter()
for v in dupAfter.values():
    k = sorted(dateKind(r) for r in v)
    if any(x != "válida" for x in k):
        kinds[(" + ".join(k), "con Update" if any(map(isUpdate, v)) else "sin Update")] += 1

winners = {k: winnerVote(v) for k, v in dupAfter.items()}
withUpdate = {k: v for k, v in dupAfter.items() if any(map(isUpdate, v))}
updateLoses = [k for k in withUpdate if not isUpdate(winners[k])]
circaVsExact = sum(1 for v in dupAfter.values() if {"circa", "válida"} <= {dateKind(r) for r in v})

display(pd.DataFrame([
    ("pares duplicados antes de reasignar (B1)", len(dupBefore)),
    ("pares duplicados después de reasignar", len(dupAfter)),
    ("· nuevos o agrandados por votos en 2+ copias", len(fromCopies)),
    ("· de esos, mismo email y nombre en todos los votos",
     sum(len({(r["user_email"], r["user_name"]) for r in v}) == 1 for v in fromCopies.values())),
    ("pares con 'Update'", len(withUpdate)),
    ("· el 'Update' pierde (tiene fecha futura)", len(updateLoses)),
    ("pares con 'circa' frente a fecha exacta (límite de B13)", circaVsExact),
    ("votos que van a soft delete", sum(len(v) - 1 for v in dupAfter.values())),
    ("votos identificados que quedan (uno por usuario+película)", len(votesAfter)),
], columns=["chequeo", "valor"]).set_index("chequeo"))
display(pd.DataFrame([(k, u, n) for (k, u), n in kinds.most_common()], columns=["fechas del par", "Update", "pares"]))

caseUser = next(k for k, v in dupAfter.items() if len(v) == 3)
pd.DataFrame([(r["movie_id"], r["rating"], str(r["date"]), dateKind(r), r["comment"][:50], r is winners[caseUser])
              for r in dupAfter[caseUser]], columns=["movie_id", "rating", "date", "fecha", "comment", "gana"])

,valor
chequeo,
pares duplicados antes de reasignar (B1),448
pares duplicados después de reasignar,472
· nuevos o agrandados por votos en 2+ copias,25
"· de esos, mismo email y nombre en todos los votos",25
pares con 'Update',222
· el 'Update' pierde (tiene fecha futura),16
pares con 'circa' frente a fecha exacta (límite de B13),0
votos que van a soft delete,473
votos identificados que quedan (uno por usuario+película),60556


,fechas del par,Update,pares
0,futura + válida,con Update,15
1,futura + válida,sin Update,13
2,circa + circa,sin Update,2
3,futura + válida + válida,con Update,1
4,sin fecha + sin fecha,con Update,1
5,circa + circa,con Update,1


,movie_id,rating,date,fecha,comment,gana
0,573a1396f29313caabce36e1,2,2026-11-19 06:05:22,futura,Update: Changed my mind after rewatching. Impressi,False
1,573a13a0f29313caabd04993,5,2026-07-06 16:33:47,válida,asdf,False
2,573a1396f29313caabce36e1,4,2026-10-29 06:05:22,válida,Impressive direction and great creative vision thr,True


### 5.3 Reseñas sin nada útil después de las reglas (B12) y ratings como texto (B14)

B5 convierte los ratings fuera de 1–5 en `null`. Una reseña queda **sin nada útil** si además el texto no dice "N stars"
(B6) y el comentario es basura o de bajo contenido (B7/B8): no hay voto ni texto del que inferirlo. B12 decidió soft
delete para esos casos; acá se cuentan.

**Decisión B14:** los ratings guardados como texto `'1'` a `'5'` (873) se **convierten a número**: es el mismo valor con
otro tipo (mismo criterio que A14 con `imdb.votes`).

**Resultado:** B12 tiene **0 casos** en este dataset: todas las reseñas con rating inválido tienen un comentario útil. La
regla se mantiene como protección (no cuesta nada y cubre datos futuros) y se documenta que se buscó y no se encontró.

In [67]:
def ratingAfterRules(r):
    """Rating válido después de B5 y B14: entero 1–5 o None."""
    if isinstance(r, str) and r.strip() in {"1", "2", "3", "4", "5"}:
        return int(r)  # B14
    if isinstance(r, int) and not isinstance(r, bool) and 1 <= r <= 5:
        return r
    return None  # B5: null, fuera de rango

def starsInText(text):
    found = STARS.search(text) if isinstance(text, str) else None
    return int(found.group(1)) if found and 1 <= int(found.group(1)) <= 5 else None

def invalidRatingDestination(r):
    if starsInText(r["comment"]) is not None:
        return "lo rescata 'N stars' (B6)"
    if commentQuality(r["comment"]) == "útil":
        return "comentario útil → se infiere con LLM"
    return "comentario basura o de bajo contenido → B12 (soft delete)"

invalidRating = [r for r in reviews if ratingAfterRules(r["rating"]) is None]
destinations = Counter(invalidRatingDestination(r) for r in invalidRating)
for destination in ["lo rescata 'N stars' (B6)", "comentario útil → se infiere con LLM",
                    "comentario basura o de bajo contenido → B12 (soft delete)"]:
    destinations.setdefault(destination, 0)

print("Ratings como texto convertidos (B14):", sum(isinstance(r["rating"], str) and ratingAfterRules(r["rating"]) is not None for r in reviews))
print("Reseñas con rating inválido después de B5:", len(invalidRating))
pd.Series(destinations, name="reseñas").to_frame()

Ratings como texto convertidos (B14): 873
Reseñas con rating inválido después de B5: 2139


,reseñas
comentario útil → se infiere con LLM,1894
lo rescata 'N stars' (B6),245
comentario basura o de bajo contenido → B12 (soft delete),0


### 5.4 Reseñas con varias anomalías a la vez

Se marca cada reseña con sus anomalías (sin contar cambios de formato, como `'4'` en vez de `4` o fechas en texto) y se
cuentan las combinaciones. "Voto duplicado" se mide **después** de reasignar las reseñas de películas duplicadas (5.2).

**Resultado:** ninguna reseña tiene más de 2 anomalías, y todas las combinaciones salvo una incluyen "voto duplicado":
los problemas se inyectaron **por separado** y solo se cruzan cuando un voto duplicado copia una reseña que ya tenía otro
problema. La excepción (anónima + fecha futura) no genera conflicto: va a soft delete por B4.

**Consecuencias conocidas de B13** (se mantiene tal cual; decisión del usuario): como el ganador se elige por fecha,
a veces es "peor" que el perdedor, porque el ruido le tocó por casualidad a la copia más reciente:
- **1 caso** (`usr_0263`): el ganador tiene rating inválido (6 → `null` → LLM) y el perdedor un 5 válido.
- **2 casos** (`usr_0099`, `usr_0735`): el ganador tiene comentario basura (se oculta, B7) y el perdedor uno útil.

Se descartaron: pasar al siguiente voto si el comentario es basura (en `usr_0735` cambia un rating válido y actual por
uno roto) y preferir el voto más completo antes que la fecha (contradice "la fecha manda"). No se mezclan campos de
votos distintos: un voto es lo que una persona dijo en un momento, y mezclarlo inventa una reseña que no existió.

In [68]:
duplicateVoteIds = {r["_id"] for v in dupAfter.values() for r in v}

def reviewAnomalies(r):
    found = set()
    if r["movie_id"] not in movieById:
        found.add("huérfana")
    if r["user_id"] in ANONYMOUS_IDS:
        found.add("anónima")
    if r["rating"] is None:
        found.add("rating null")
    elif ratingAfterRules(r["rating"]) is None:
        found.add("rating fuera de rango")
    if commentQuality(r["comment"]) != "útil":
        found.add(f"comentario {commentQuality(r['comment'])}")
    if r["date"] == "yesterday":
        found.add("fecha 'yesterday'")
    elif str(r["date"]).startswith("circa"):
        found.add("fecha 'circa'")
    elif (parsed := parseReviewDate(r["date"])) and parsed > datasetCreated:
        found.add("fecha futura")
    if r["_id"] in duplicateVoteIds:
        found.add("voto duplicado")
    return found

anomaliesPerReview = [reviewAnomalies(r) for r in reviews]
display(pd.Series(Counter(len(a) for a in anomaliesPerReview), name="reseñas").sort_index().rename_axis("anomalías").to_frame().T)
display(pd.Series(Counter(" + ".join(sorted(a)) for a in anomaliesPerReview if len(a) > 1), name="reseñas")
        .sort_values(ascending=False).rename_axis("combinación").to_frame())

# Ganadores de B13 que son "peores" que algún perdedor del mismo par
worseWinner = []
for key, votes in dupAfter.items():
    w = winners[key]
    losers = [r for r in votes if r is not w]
    if ratingAfterRules(w["rating"]) is None and any(ratingAfterRules(r["rating"]) is not None for r in losers):
        worseWinner.append(("rating inválido; el perdedor tenía uno válido", key[0], votes, w))
    if commentQuality(w["comment"]) != "útil" and any(commentQuality(r["comment"]) == "útil" for r in losers):
        worseWinner.append(("comentario basura; el perdedor tenía uno útil", key[0], votes, w))

pd.DataFrame([(case, user, r["rating"], str(r["date"])[:10], r["comment"][:55], r is w)
              for case, user, votes, w in worseWinner for r in votes],
             columns=["caso", "user_id", "rating", "date", "comment", "gana"])

anomalías,0,1,2
reseñas,55688,5713,87


,reseñas
combinación,
fecha futura + voto duplicado,29
comentario basura + voto duplicado,24
rating null + voto duplicado,10
rating fuera de rango + voto duplicado,9
comentario bajo contenido + voto duplicado,6
fecha 'circa' + voto duplicado,6
fecha 'yesterday' + voto duplicado,2
anónima + fecha futura,1


,caso,user_id,rating,date,comment,gana
0,comentario basura; el perdedor tenía uno útil,usr_0099,5,2025-11-08,movie,True
1,comentario basura; el perdedor tenía uno útil,usr_0099,5,2023-08-23,Simply unforgettable from the very first frame to the c,False
2,comentario basura; el perdedor tenía uno útil,usr_0735,0,2024-01-03,Definitely worth watching on a weekend night. The story,False
3,comentario basura; el perdedor tenía uno útil,usr_0735,2,2025-02-08,...,True
4,rating inválido; el perdedor tenía uno válido,usr_0263,6,2026-07-03,A breathtaking achievement that completely blew me away,True
5,rating inválido; el perdedor tenía uno válido,usr_0263,5,2024-06-04,An absolute cinematic masterpiece. The performances del,False
